# ATML PA1 Task 2 — Corrected UDA rerun

## Provenance

This is a corrected rerun of Task 2. The previous attempt is preserved on the GitHub branch
`archive/task2-attempt-1-20260923`. Earlier target results are known, but they will not be used
to change the corrected protocol, choose checkpoints, or tune the new runs.

## Assignment-mandated protocol

### Dataset and label access

- Dataset: PACS with the seven shared classes.
- Labeled source domains: Photo, Art Painting, and Cartoon.
- Target domain: Sketch.
- All Sketch images may be used during Task 2 adaptation without class labels.
- Sketch class labels may be accessed only after every model, setting, and checkpoint is fixed.
- Use the same source splits in Tasks 2 and 3.
- The Task 2 Source-only checkpoint must be reused unchanged as the Task 3 ERM baseline.
- Task 2 target results must not influence Task 3 methods or settings.

### Source protocol

- Within each source domain, use a stratified 80/20 train/validation split with seed 6304.
- Use source training labels for optimization.
- Select checkpoints using the unweighted mean macro-F1 across the three source validation domains.
- Reuse the already verified `shared/splits/pacs_sketch_seed6304.json` manifest.

### Model and preprocessing

- Use torchvision ResNet-18 with `ResNet18_Weights.IMAGENET1K_V1`.
- Replace the ImageNet classifier with a seven-class linear head.
- Fine-tune the complete network for every method.
- Resize images to 256 × 256.
- During training, use a random 224 × 224 crop and horizontal flipping.
- During validation and evaluation, use a 224 × 224 center crop.
- Apply the normalization associated with the pretrained weights.
- Freeze BatchNorm running means and variances at their pretrained ImageNet values.
- Keep BatchNorm gamma and beta parameters trainable.
- After `model.train()`, place only BatchNorm modules in evaluation mode.

### Optimization and batching

- Use AdamW with learning rate 1e-4 and weight decay 1e-4.
- Train for at most 30 source epochs.
- Stop after five epochs without improved mean source-validation macro-F1.
- Use seed 6304 for every comparison.
- Each adaptation update must contain 8 examples from each source domain and 24 target examples.
- Cycle loaders when needed.
- Keep initialization, source sampling, augmentation, optimizer, training budget, and evaluation pipeline fixed across methods.

### Required methods

- Source-only: source cross-entropy with domain-balanced source batches.
- DAN: MMD on the raw 512-dimensional feature immediately before the classifier.
- Main DAN weight: lambda_MMD = 1.
- DAN kernels: three RBF kernels with bandwidth factors 0.5, 1, and 2 times the median pairwise squared distance in the current combined batch.
- DANN discriminator: 512 → 256 → 2 with ReLU and dropout 0.5.
- DANN uses the required gradient-reversal schedule and unit domain-loss weight.
- Only source examples contribute to classification loss.
- Source and target examples both contribute to domain loss.
- CDAN conditions the discriminator on `vec(feature outer-product class_probability)`, giving 3584 inputs.
- CDAN uses the same discriminator, schedule, and loss weight as DANN.
- CDAN must not use entropy conditioning or detach its feature or probability inputs.

### TA clarification for DAN bandwidth calculation

- Remove diagonal self-distances when calculating the median pairwise squared distance.
- Retain off-diagonal zero distances.
- Do not silently remove additional zero distances.
- If the resulting median creates a numerical problem, stop and document it before changing the convention.

### Evaluation

- After checkpoints are frozen, report per-source validation accuracy and macro-F1.
- Report mean source accuracy and macro-F1.
- Report Sketch accuracy, macro-F1, and accuracy change relative to Source-only.
- Freeze each backbone for the domain-separability diagnostic.
- Use equal source-validation and target feature counts.
- Use seed 6304, a 70/30 split, and balanced logistic regression with C=1.
- Domain separability chance performance is 50%.
- After freezing, compute per-class Sketch accuracy changes and inspect dominant confusions.

### Controlled study

Choose exactly one:

1. DAN with lambda_MMD in {0.1, 1, 10}; or
2. DANN with maximum GRL strength in {0.25, 0.5, 1}.

All other settings must remain fixed. Expectations must be written before interpreting target results.
Target results cannot be used to revise the tested settings.

### Required evidence

- Complete comparison table for Source-only, DAN, DANN, and CDAN.
- Classification and alignment/domain-loss curves.
- Per-class Sketch changes and selected confusions or failure cases.
- Compact controlled-strength table or plot.
- Code, configurations, split indices, environment, machine-readable results, and reproduction instructions in the public GitHub repository.
- Raw datasets and unnecessary large checkpoints must remain outside Git.

## Decision rule for this rerun

Anything not explicitly mandated above must be entered in the decision ledger and approved before it is implemented.
No hyperparameter, numerical convention, stabilization method, library default, or post-failure adjustment may be
introduced silently.

In [ ]:
# CELL 1 — clean-run environment preflight only

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import json
import platform
import torch

ASSIGNMENT_SEED = 6304
RUN_ID = "task2_corrected_v1_20260923"

RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1") / RUN_ID
PREFLIGHT_DIR = RUN_ROOT / "preflight"
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab select Runtime → Change runtime type → GPU, "
        "then rerun this cell."
    )

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

environment = {
    "run_id": RUN_ID,
    "recorded_utc": datetime.now(timezone.utc).isoformat(),
    "assignment_seed": ASSIGNMENT_SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": package_version("torchvision"),
    "numpy": package_version("numpy"),
    "scikit_learn": package_version("scikit-learn"),
    "pillow": package_version("Pillow"),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capability": list(torch.cuda.get_device_capability(0)),
    "run_root": str(RUN_ROOT),
    "stage": "environment_preflight_only",
    "dataset_loaded": False,
    "target_labels_accessed": False,
    "training_started": False,
}

record_path = PREFLIGHT_DIR / "environment_preflight.json"
record_path.write_text(json.dumps(environment, indent=2) + "\n")

print(json.dumps(environment, indent=2))
print("\nPreflight record:", record_path)
print("No dataset, model, target label, or training code has run.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "run_id": "task2_corrected_v1_20260923",
  "recorded_utc": "2026-09-23T11:04:19.343574+00:00",
  "assignment_seed": 6304,
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "numpy": "2.1.3",
  "scikit_learn": "1.6.1",
  "pillow": "11.3.0",
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu": "Tesla T4",
  "gpu_capability": [
    7,
    5
  ],
  "run_root": "/content/drive/MyDrive/ATML-PA1/task2_corrected_v1_20260923",
  "stage": "environment_preflight_only",
  "dataset_loaded": false,
  "target_labels_accessed": false,
  "training_started": false
}

Preflight record: /content/drive/MyDrive/ATML-PA1/task2_corrected_v1_20260923/preflight/environment_preflight.json
No dataset, model, target label, or training code has run.


In [ ]:
# CELL 1 — fresh runtime, exact-code verification, and local tests.
# This does NOT access PACS or start training.

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import shutil
import hashlib
import zipfile
import platform
import compileall
import importlib.util
from pathlib import Path

import torch
import torchvision
import numpy as np
import sklearn

from google.colab import drive, files

EXPECTED_ZIP_SHA256 = "f85cee7f58ccad18d9820bd11b685c1463a29ad0130042c35adc3d73c98a0a6e"
ARCHIVE_NAME = "ATML-PA1-Task2-corrected-code-v1.zip"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU unavailable. Select Runtime → Change runtime type → GPU, then rerun."
    )

drive.mount("/content/drive")

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one file: " + ARCHIVE_NAME)

uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
actual_zip_sha256 = hashlib.sha256(uploaded_bytes).hexdigest()

if actual_zip_sha256 != EXPECTED_ZIP_SHA256:
    raise RuntimeError(
        "Wrong or modified ZIP.\n"
        f"Uploaded: {uploaded_name}\n"
        f"Expected SHA256: {EXPECTED_ZIP_SHA256}\n"
        f"Actual SHA256:   {actual_zip_sha256}"
    )

archive_path = Path("/content") / ARCHIVE_NAME
archive_path.write_bytes(uploaded_bytes)

code_root = Path("/content/task2-corrected-code")
temporary_root = Path("/content/task2-code-extraction")

if temporary_root.exists():
    shutil.rmtree(temporary_root)

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(temporary_root)

extracted_root = temporary_root / "task2-corrected-code"
manifest = json.loads(
    (extracted_root / "PACKAGE-MANIFEST.json").read_text()
)

for record in manifest["files"]:
    path = extracted_root / record["path"]
    if not path.is_file():
        raise RuntimeError(f"Package file missing: {record['path']}")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != record["sha256"]:
        raise RuntimeError(f"Package file changed: {record['path']}")

if code_root.exists():
    shutil.rmtree(code_root)

shutil.move(str(extracted_root), str(code_root))
shutil.rmtree(temporary_root)

if not compileall.compile_dir(code_root, quiet=1, force=True):
    raise RuntimeError("Python compilation failed")

sys.path.insert(0, str(code_root))

test_path = code_root / "tests" / "test_locked_choices.py"
spec = importlib.util.spec_from_file_location("locked_tests", test_path)
locked_tests = importlib.util.module_from_spec(spec)
spec.loader.exec_module(locked_tests)

tests_run = []
for name in sorted(dir(locked_tests)):
    if name.startswith("test_"):
        getattr(locked_tests, name)()
        tests_run.append(name)

run_root = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923"
)
code_archive_root = run_root / "code"
code_archive_root.mkdir(parents=True, exist_ok=True)

saved_archive = code_archive_root / ARCHIVE_NAME
if saved_archive.exists():
    saved_hash = hashlib.sha256(saved_archive.read_bytes()).hexdigest()
    if saved_hash != EXPECTED_ZIP_SHA256:
        raise RuntimeError("A different archive already exists in the corrected-run folder")
else:
    saved_archive.write_bytes(uploaded_bytes)

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
    "numpy": str(np.__version__),
    "sklearn": str(sklearn.__version__),
    "cuda": str(torch.version.cuda),
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "package_sha256": EXPECTED_ZIP_SHA256,
}

environment_path = run_root / "environment_preflight.json"
if environment_path.exists():
    previous = json.loads(environment_path.read_text())
    if previous != environment:
        raise RuntimeError(
            "Runtime differs from the previously recorded corrected-run environment."
        )
else:
    environment_path.write_text(json.dumps(environment, indent=2) + "\n")

print(json.dumps(environment, indent=2))
print(f"\nCODE VERIFIED: {len(manifest['files'])} manifested files")
print(f"TESTS PASSED: {len(tests_run)}")
print("TRAINING STARTED: False")
print("TARGET LABELS ACCESSED: False")
print("Code root:", code_root)
print("Persistent run root:", run_root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving ATML-PA1-Task2-corrected-code-v1.zip to ATML-PA1-Task2-corrected-code-v1.zip
{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "numpy": "2.1.3",
  "sklearn": "1.6.1",
  "cuda": "12.8",
  "cudnn": 91900,
  "gpu": "Tesla T4",
  "gpu_count": 1,
  "package_sha256": "f85cee7f58ccad18d9820bd11b685c1463a29ad0130042c35adc3d73c98a0a6e"
}

CODE VERIFIED: 25 manifested files
TESTS PASSED: 6
TRAINING STARTED: False
TARGET LABELS ACCESSED: False
Code root: /content/task2-corrected-code
Persistent run root: /content/drive/MyDrive/ATML-PA1/task2_corrected_20260923


In [ ]:
# CELL 2 — acquire and verify PACS, then run the read-only data preflight.

import os
import sys
import json
import shutil
import hashlib
import zipfile
import subprocess
from pathlib import Path

import gdown

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")

PACS_FILE_ID = "1m4X4fROCCXMO0lRLrr6Zz9Vb3974NWhE"
PACS_URL = f"https://drive.google.com/uc?id={PACS_FILE_ID}"
EXPECTED_PACS_SHA256 = (
    "0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
)
EXPECTED_ARCHIVE_MEMBERS = 10044

LOCAL_ARCHIVE = Path("/content/PACS_dassl.zip")
DRIVE_ARCHIVE = (
    Path("/content/drive/MyDrive/ATML-PA1/datasets/PACS_dassl.zip")
)
EXTRACTION_TEMP = Path("/content/pacs_extraction_temp")
EXTRACTION_ROOT = Path("/content/atml_pacs")
PACS_ROOT = EXTRACTION_ROOT / "pacs" / "images"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

DRIVE_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_ARCHIVE.exists():
    stored_hash = sha256_file(DRIVE_ARCHIVE)
    if stored_hash != EXPECTED_PACS_SHA256:
        raise RuntimeError(
            "The PACS archive already stored in Drive has the wrong hash.\n"
            f"Expected: {EXPECTED_PACS_SHA256}\n"
            f"Actual:   {stored_hash}"
        )
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
    print("Using the verified PACS archive stored in Drive.")
else:
    print("Downloading the pinned PACS archive...")
    gdown.download(
        id=PACS_FILE_ID,
        output=str(LOCAL_ARCHIVE),
        quiet=False,
    )

    downloaded_hash = sha256_file(LOCAL_ARCHIVE)
    if downloaded_hash != EXPECTED_PACS_SHA256:
        raise RuntimeError(
            "Downloaded PACS archive has the wrong hash.\n"
            f"Expected: {EXPECTED_PACS_SHA256}\n"
            f"Actual:   {downloaded_hash}"
        )

    shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)
    print("Verified PACS archive saved to Drive.")

archive_hash = sha256_file(LOCAL_ARCHIVE)
if archive_hash != EXPECTED_PACS_SHA256:
    raise RuntimeError("Local PACS archive verification failed.")

if EXTRACTION_TEMP.exists():
    shutil.rmtree(EXTRACTION_TEMP)
EXTRACTION_TEMP.mkdir(parents=True)

with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
    members = archive.infolist()

    if len(members) != EXPECTED_ARCHIVE_MEMBERS:
        raise RuntimeError(
            f"Unexpected archive member count: {len(members)}"
        )

    extraction_base = EXTRACTION_TEMP.resolve()
    for member in members:
        destination = (EXTRACTION_TEMP / member.filename).resolve()
        if (
            destination != extraction_base
            and extraction_base not in destination.parents
        ):
            raise RuntimeError(
                f"Unsafe path found in archive: {member.filename}"
            )

    archive.extractall(EXTRACTION_TEMP)

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)
shutil.move(str(EXTRACTION_TEMP), str(EXTRACTION_ROOT))

if not PACS_ROOT.is_dir():
    raise RuntimeError(f"Expected PACS directory not found: {PACS_ROOT}")

preflight_output = RUN_ROOT / "preflight" / "data_preflight.json"
preflight_output.parent.mkdir(parents=True, exist_ok=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(CODE_ROOT)

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared"
            / "splits"
            / "pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_output),
    ],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

result = json.loads(preflight_output.read_text())

print("\nPACS ARCHIVE SHA256:", archive_hash)
print("PACS ROOT:", PACS_ROOT)
print("PREFLIGHT STATUS:", result["status"])
print("TRAINING STARTED:", result["training_started"])
print("TARGET LABELS ACCESSED:", result["target_labels_accessed"])
print("STEPS PER SOURCE EPOCH:", result["steps_per_source_epoch"])

Using the verified PACS archive stored in Drive.

PACS ARCHIVE SHA256: 0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102
PACS ROOT: /content/atml_pacs/pacs/images
PREFLIGHT STATUS: PREFLIGHT_PASS
TRAINING STARTED: False
TARGET LABELS ACCESSED: False
STEPS PER SOURCE EPOCH: 235


In [ ]:
# CELL 3 — lock the student's expectation and create the common initialization.

import os
import sys
import shutil
import hashlib
import subprocess
from pathlib import Path

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")

EXPECTATION = """
I expect the following:

Increasing lambda_MMD and source performance:
As lambda_MMD increases from 0.1 to 1 to 10, the MMD penalty places stronger pressure on the 512-dimensional feature to make source and target marginal distributions similar. Because only source examples contribute to the classification loss, this alignment pressure should compete with class-discriminative source training. I expect mean source-validation accuracy and macro-F1 to decrease monotonically with lambda_MMD: the smallest drop at 0.1, a modest drop at 1, and the largest drop at 10. At lambda_MMD=10, excessive marginal alignment may noticeably weaken source classification.

Domain separability:
I expect the source-vs-target logistic-regression separability score to decrease as lambda_MMD increases. At lambda_MMD=0.1, source and target features should remain fairly distinguishable, so separability should be well above 50%. At lambda_MMD=1, separability should drop meaningfully. At lambda_MMD=10, it should approach chance or become very low, indicating strong marginal alignment. However, lower separability alone would not prove that class information is preserved.

Target recognition:
I expect a non-monotonic pattern. Weak alignment at lambda_MMD=0.1 should behave close to Source-only ERM because the penalty is too small to close the domain gap. Moderate alignment at lambda_MMD=1 should give the best target accuracy and macro-F1, reducing domain shift without destroying class structure. Strong alignment at lambda_MMD=10 should over-align the marginals, mix class-discriminative information, and cause negative transfer, so target recognition should fall below the lambda_MMD=1 result and possibly below ERM. Overall, I expect the best target performance at lambda_MMD=1, with an inverted-U relationship between alignment strength and target recognition.

Disclosure:
Before locking this corrected rerun, I had already seen the target-domain results from the archived first attempt. Those results were not used to change the approved methods, hyperparameters, source split, checkpoint-selection rule, or evaluation protocol used for this rerun.
""".strip()

I_CONFIRM_THIS_TEXT_IS_MY_OWN = True

I_CONFIRM_THIS_TEXT_IS_MY_OWN = True

if not I_CONFIRM_THIS_TEXT_IS_MY_OWN:
    raise RuntimeError(
        "Set I_CONFIRM_THIS_TEXT_IS_MY_OWN = True after writing your statement."
    )

if (
    not EXPECTATION
    or "REPLACE THIS" in EXPECTATION
    or "PENDING" in EXPECTATION.upper()
):
    raise RuntimeError("Replace the placeholder with your own statement.")

if len(EXPECTATION) < 100:
    raise RuntimeError(
        "The statement is too short to cover source performance, "
        "domain separability, target recognition, and the disclosure."
    )

drive_preregistration = (
    RUN_ROOT
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)
code_preregistration = (
    CODE_ROOT
    / "task2"
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)

drive_preregistration.parent.mkdir(parents=True, exist_ok=True)

text_to_save = EXPECTATION.strip() + "\n"
drive_preregistration.write_text(text_to_save)
code_preregistration.write_text(text_to_save)

preregistration_sha256 = hashlib.sha256(
    text_to_save.encode("utf-8")
).hexdigest()

initialization_path = (
    RUN_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)
initialization_path.parent.mkdir(parents=True, exist_ok=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(CODE_ROOT)

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.prepare_initialization",
        "--output",
        str(initialization_path),
    ],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

print("\nLOCKED EXPECTATION:")
print(text_to_save)
print("EXPECTATION SHA256:", preregistration_sha256)
print("COMMON INITIALIZATION:", initialization_path)
print("TRAINING STARTED: False")
print("TARGET LABELS ACCESSED: False")


LOCKED EXPECTATION:
I expect the following:

Increasing lambda_MMD and source performance:
As lambda_MMD increases from 0.1 to 1 to 10, the MMD penalty places stronger pressure on the 512-dimensional feature to make source and target marginal distributions similar. Because only source examples contribute to the classification loss, this alignment pressure should compete with class-discriminative source training. I expect mean source-validation accuracy and macro-F1 to decrease monotonically with lambda_MMD: the smallest drop at 0.1, a modest drop at 1, and the largest drop at 10. At lambda_MMD=10, excessive marginal alignment may noticeably weaken source classification.

Domain separability:
I expect the source-vs-target logistic-regression separability score to decrease as lambda_MMD increases. At lambda_MMD=0.1, source and target features should remain fairly distinguishable, so separability should be well above 50%. At lambda_MMD=1, separability should drop meaningfully. At lambda_

In [ ]:
# CELL 4 — train or safely resume the Source-only baseline.

import os
import sys
import json
import subprocess
from pathlib import Path

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")
RUNS_ROOT = RUN_ROOT / "runs"

PACS_ROOT = Path("/content/atml_pacs/pacs/images")
PROTOCOL = (
    CODE_ROOT / "shared" / "splits" / "pacs_sketch_seed6304.json"
)
INITIALIZATION = (
    RUN_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    RUN_ROOT / "preregistration" / "DAN_STRENGTH_EXPECTATION.txt"
)

DATASET_SOURCE = (
    "Google Drive file 1m4X4fROCCXMO0lRLrr6Zz9Vb3974NWhE; "
    "SHA256 0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
)

for required_path in (
    CODE_ROOT,
    PACS_ROOT,
    PROTOCOL,
    INITIALIZATION,
    PREREGISTRATION,
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required path missing: {required_path}")

if not __import__("torch").cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Reconnect to a GPU runtime.")

run_directory = RUNS_ROOT / "source_only"
finished_manifest = run_directory / "run.json"
resume_checkpoint = run_directory / "last.pt"

if finished_manifest.exists():
    print("Source-only is already complete; no training was repeated.")
    print(json.dumps(json.loads(finished_manifest.read_text()), indent=2))
else:
    command = [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id",
        "source_only",
        "--pacs-root",
        str(PACS_ROOT),
        "--dataset-source",
        DATASET_SOURCE,
        "--protocol",
        str(PROTOCOL),
        "--initialization",
        str(INITIALIZATION),
        "--preregistration",
        str(PREREGISTRATION),
        "--output",
        str(RUNS_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
    ]

    if resume_checkpoint.exists():
        command.append("--resume")
        print("Resuming Source-only from its last complete epoch.")
    elif run_directory.exists():
        raise RuntimeError(
            "The Source-only directory exists without a complete-epoch checkpoint. "
            "Stop here and show me the directory contents."
        )
    else:
        print("Starting Source-only from the locked common initialization.")

    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(CODE_ROOT)
    environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    subprocess.run(
        command,
        cwd=CODE_ROOT,
        env=environment,
        check=True,
    )

manifest = json.loads(finished_manifest.read_text())

print("\nSOURCE-ONLY VERIFIED COMPLETE")
print("Epochs completed:", manifest["epochs_completed"])
print("Selected epoch:", manifest["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    manifest["best_mean_source_macro_f1"],
)
print("Checkpoint SHA256:", manifest["best_checkpoint_sha256"])
print("Target labels used:", manifest["target_labels_used"])

Starting Source-only from the locked common initialization.

SOURCE-ONLY VERIFIED COMPLETE
Epochs completed: 11
Selected epoch: 6
Best mean source-validation macro-F1: 0.9373899735024244
Checkpoint SHA256: 034a4bb4035184e98b950d44f5afa13003a7e6d44f652f668c138cce71e8a552
Target labels used: False


In [ ]:
# CELL 5 — train and verify DAN with lambda_MMD = 0.1.

import os
import sys
import csv
import json
import math
import hashlib
import subprocess
from pathlib import Path

import torch

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")
RUNS_ROOT = RUN_ROOT / "runs"

PACS_ROOT = Path("/content/atml_pacs/pacs/images")
PROTOCOL = CODE_ROOT / "shared" / "splits" / "pacs_sketch_seed6304.json"
INITIALIZATION = (
    RUN_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    RUN_ROOT / "preregistration" / "DAN_STRENGTH_EXPECTATION.txt"
)

DATASET_SOURCE = (
    "Google Drive file 1m4X4fROCCXMO0lRLrr6Zz9Vb3974NWhE; "
    "SHA256 0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
)

RUN_ID = "dan_0p1"
run_directory = RUNS_ROOT / RUN_ID
finished_manifest = run_directory / "run.json"
resume_checkpoint = run_directory / "last.pt"

if finished_manifest.exists():
    print(f"{RUN_ID} is already complete; training was not repeated.")
else:
    command = [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id", RUN_ID,
        "--pacs-root", str(PACS_ROOT),
        "--dataset-source", DATASET_SOURCE,
        "--protocol", str(PROTOCOL),
        "--initialization", str(INITIALIZATION),
        "--preregistration", str(PREREGISTRATION),
        "--output", str(RUNS_ROOT),
        "--device", "cuda",
        "--num-workers", "2",
    ]

    if resume_checkpoint.exists():
        command.append("--resume")
        print(f"Resuming {RUN_ID} from its last complete epoch.")
    elif run_directory.exists():
        raise RuntimeError(
            f"{run_directory} exists without a resumable checkpoint. Stop here."
        )
    else:
        print("Starting DAN lambda_MMD=0.1 from the common initialization.")

    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(CODE_ROOT)
    environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    subprocess.run(
        command,
        cwd=CODE_ROOT,
        env=environment,
        check=True,
    )

manifest = json.loads(finished_manifest.read_text())

with (run_directory / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

if len(history) != manifest["epochs_completed"]:
    raise RuntimeError("History length does not match the run manifest.")

numeric_fields = [
    field
    for field in history[0]
    if field != "epoch"
]

for row in history:
    for field in numeric_fields:
        if not math.isfinite(float(row[field])):
            raise RuntimeError(
                f"Non-finite history value at epoch {row['epoch']}: {field}"
            )

if any(
    float(row["mmd_median_squared_distance"]) <= 0
    for row in history
):
    raise RuntimeError("A non-positive MMD bandwidth median was recorded.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

best_checkpoint = run_directory / "best.pt"
if sha256_file(best_checkpoint) != manifest["best_checkpoint_sha256"]:
    raise RuntimeError("Best checkpoint hash does not match the manifest.")

initial_state = torch.load(
    INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)["state_dict"]

best_payload = torch.load(
    best_checkpoint,
    map_location="cpu",
    weights_only=False,
)

if best_payload["target_labels_used"] is not False:
    raise RuntimeError("Target-label isolation assertion failed.")

bn_names = [
    name for name in initial_state
    if name.endswith(("running_mean", "running_var", "num_batches_tracked"))
]

changed_bn = [
    name
    for name in bn_names
    if not torch.equal(
        initial_state[name],
        best_payload["model_state"][name],
    )
]

if changed_bn:
    raise RuntimeError(
        f"BatchNorm buffers changed unexpectedly: {changed_bn[:3]}"
    )

print("\nDAN lambda_MMD=0.1 VERIFIED COMPLETE")
print("Epochs completed:", manifest["epochs_completed"])
print("Selected epoch:", manifest["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    manifest["best_mean_source_macro_f1"],
)
print(
    "Mean recorded MMD loss:",
    sum(float(row["mmd_loss"]) for row in history) / len(history),
)
print(
    "Maximum mean off-diagonal zero count:",
    max(float(row["mmd_off_diagonal_zero_count"]) for row in history),
)
print("Checkpoint SHA256:", manifest["best_checkpoint_sha256"])
print("BatchNorm buffers unchanged:", not changed_bn)
print("Target labels used:", manifest["target_labels_used"])

Starting DAN lambda_MMD=0.1 from the common initialization.

DAN lambda_MMD=0.1 VERIFIED COMPLETE
Epochs completed: 10
Selected epoch: 5
Best mean source-validation macro-F1: 0.9483037086522582
Mean recorded MMD loss: 0.16279880270044855
Maximum mean off-diagonal zero count: 0.0
Checkpoint SHA256: 92048571aa48cf1e29a03141520a06aa945aabdb9b492ea54822109a3f8863a5
BatchNorm buffers unchanged: True
Target labels used: False


In [ ]:
# CELL 6 — reusable locked trainer; run DAN with lambda_MMD = 1.

import os
import sys
import csv
import json
import math
import hashlib
import subprocess
from pathlib import Path

import torch

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")
RUNS_ROOT = RUN_ROOT / "runs"
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
PROTOCOL = CODE_ROOT / "shared" / "splits" / "pacs_sketch_seed6304.json"
INITIALIZATION = RUN_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
PREREGISTRATION = (
    RUN_ROOT / "preregistration" / "DAN_STRENGTH_EXPECTATION.txt"
)

DATASET_SOURCE = (
    "Google Drive file 1m4X4fROCCXMO0lRLrr6Zz9Vb3974NWhE; "
    "SHA256 0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def train_and_verify(run_id):
    run_directory = RUNS_ROOT / run_id
    finished_manifest = run_directory / "run.json"
    resume_checkpoint = run_directory / "last.pt"

    if finished_manifest.exists():
        print(f"{run_id} is already complete; training was not repeated.")
    else:
        command = [
            sys.executable,
            "-m", "task2.train",
            "--run-id", run_id,
            "--pacs-root", str(PACS_ROOT),
            "--dataset-source", DATASET_SOURCE,
            "--protocol", str(PROTOCOL),
            "--initialization", str(INITIALIZATION),
            "--preregistration", str(PREREGISTRATION),
            "--output", str(RUNS_ROOT),
            "--device", "cuda",
            "--num-workers", "2",
        ]

        if resume_checkpoint.exists():
            command.append("--resume")
            print(f"Resuming {run_id} from its last complete epoch.")
        elif run_directory.exists():
            raise RuntimeError(
                f"{run_directory} exists without a resumable checkpoint."
            )
        else:
            print(f"Starting {run_id} from the locked common initialization.")

        environment = os.environ.copy()
        environment["PYTHONPATH"] = str(CODE_ROOT)
        environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

        subprocess.run(
            command,
            cwd=CODE_ROOT,
            env=environment,
            check=True,
        )

    manifest = json.loads(finished_manifest.read_text())

    with (run_directory / "history.csv").open(newline="") as handle:
        history = list(csv.DictReader(handle))

    if len(history) != manifest["epochs_completed"]:
        raise RuntimeError("History length does not match the manifest.")

    for row in history:
        for field, value in row.items():
            if field != "epoch" and not math.isfinite(float(value)):
                raise RuntimeError(
                    f"Non-finite value at epoch {row['epoch']}: {field}"
                )

    method = manifest["identity"]["config"]["method"]

    if method == "dan" and any(
        float(row["mmd_median_squared_distance"]) <= 0
        for row in history
    ):
        raise RuntimeError("A non-positive MMD median was recorded.")

    best_checkpoint = run_directory / "best.pt"
    if sha256_file(best_checkpoint) != manifest["best_checkpoint_sha256"]:
        raise RuntimeError("Best-checkpoint hash mismatch.")

    initial_state = torch.load(
        INITIALIZATION,
        map_location="cpu",
        weights_only=False,
    )["state_dict"]

    best_payload = torch.load(
        best_checkpoint,
        map_location="cpu",
        weights_only=False,
    )

    if best_payload["target_labels_used"] is not False:
        raise RuntimeError("Target-label isolation assertion failed.")

    bn_names = [
        name for name in initial_state
        if name.endswith(
            ("running_mean", "running_var", "num_batches_tracked")
        )
    ]

    changed_bn = [
        name
        for name in bn_names
        if not torch.equal(
            initial_state[name],
            best_payload["model_state"][name],
        )
    ]

    if changed_bn:
        raise RuntimeError(
            f"BatchNorm buffers changed: {changed_bn[:3]}"
        )

    print(f"\n{run_id} VERIFIED COMPLETE")
    print("Method:", method)
    print("Epochs completed:", manifest["epochs_completed"])
    print("Selected epoch:", manifest["best_epoch"])
    print(
        "Best mean source-validation macro-F1:",
        manifest["best_mean_source_macro_f1"],
    )

    if method == "dan":
        print(
            "Mean recorded MMD loss:",
            sum(float(row["mmd_loss"]) for row in history) / len(history),
        )
        print(
            "Maximum mean off-diagonal zero count:",
            max(
                float(row["mmd_off_diagonal_zero_count"])
                for row in history
            ),
        )

    if method in {"dann", "cdan"}:
        print(
            "Mean domain loss:",
            sum(float(row["domain_loss"]) for row in history) / len(history),
        )
        print(
            "Mean domain accuracy:",
            sum(float(row["domain_accuracy"]) for row in history)
            / len(history),
        )

    print("Checkpoint SHA256:", manifest["best_checkpoint_sha256"])
    print("BatchNorm buffers unchanged:", not changed_bn)
    print("Target labels used:", manifest["target_labels_used"])

    return manifest

dan_1_manifest = train_and_verify("dan_1")

Starting dan_1 from the locked common initialization.

dan_1 VERIFIED COMPLETE
Method: dan
Epochs completed: 7
Selected epoch: 2
Best mean source-validation macro-F1: 0.05066996495567924
Mean recorded MMD loss: 0.17978442534124958
Maximum mean off-diagonal zero count: 0.0
Checkpoint SHA256: bdad529a4c33c8ef1d84c5b999c216b66574055b327f2f869654be4ebdac9b0a
BatchNorm buffers unchanged: True
Target labels used: False


In [ ]:
# CELL 7 — diagnose DAN lambda=1 using source data only.
# No retraining and no Sketch labels.

import csv
import json
from collections import Counter
from pathlib import Path

import torch

from shared.pacs import (
    SOURCES,
    load_protocol,
    make_validation_loader,
)
from task2.model import PACSClassifier

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
RUN_DIRECTORY = RUN_ROOT / "runs" / "dan_1"
PROTOCOL_PATH = (
    CODE_ROOT / "shared" / "splits" / "pacs_sketch_seed6304.json"
)

with (RUN_DIRECTORY / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

print("TRAINING HISTORY — SOURCE INFORMATION ONLY")
print(
    "epoch | cls_loss | mmd_loss | weighted_total | grad_norm "
    "| photo_F1 | art_F1 | cartoon_F1 | mean_F1"
)

for row in history:
    classification_loss = float(row["classification_loss"])
    mmd_loss = float(row["mmd_loss"])

    print(
        f"{int(row['epoch']):>5} | "
        f"{classification_loss:>8.4f} | "
        f"{mmd_loss:>8.4f} | "
        f"{classification_loss + mmd_loss:>14.4f} | "
        f"{float(row['gradient_norm']):>9.4f} | "
        f"{float(row['photo_macro_f1']):>8.4f} | "
        f"{float(row['art_painting_macro_f1']):>6.4f} | "
        f"{float(row['cartoon_macro_f1']):>10.4f} | "
        f"{float(row['mean_source_macro_f1']):>7.4f}"
    )

manifest = json.loads((RUN_DIRECTORY / "run.json").read_text())
best_payload = torch.load(
    RUN_DIRECTORY / "best.pt",
    map_location="cpu",
    weights_only=False,
)

device = torch.device("cuda")
model = PACSClassifier(pretrained=False)
model.load_state_dict(best_payload["model_state"])
model.to(device)
model.eval()

protocol = load_protocol(PROTOCOL_PATH)

print("\nSELECTED-CHECKPOINT SOURCE PREDICTION COUNTS")

all_prediction_counts = Counter()

with torch.inference_mode():
    for domain in SOURCES:
        loader = make_validation_loader(
            PACS_ROOT,
            protocol["source_splits"][domain]["validation"],
            batch_size=64,
            num_workers=2,
            pin_memory=True,
        )

        truth_counts = Counter()
        prediction_counts = Counter()
        correct_counts = Counter()

        for images, labels, _ in loader:
            logits, features = model(images.to(device, non_blocking=True))
            predictions = logits.argmax(dim=1).cpu()

            for truth, prediction in zip(labels.tolist(), predictions.tolist()):
                truth_counts[int(truth)] += 1
                prediction_counts[int(prediction)] += 1
                all_prediction_counts[int(prediction)] += 1

                if truth == prediction:
                    correct_counts[int(truth)] += 1

        per_class_accuracy = {
            class_id: correct_counts[class_id] / truth_counts[class_id]
            for class_id in range(7)
        }

        print(f"\n{domain}")
        print("Prediction counts:", dict(sorted(prediction_counts.items())))
        print(
            "Per-class source-validation accuracy:",
            {
                key: round(value, 4)
                for key, value in per_class_accuracy.items()
            },
        )

print("\nPOOLED SOURCE PREDICTION COUNTS:")
print(dict(sorted(all_prediction_counts.items())))

print("\nDIAGNOSTIC COMPLETE")
print("Selected epoch:", manifest["best_epoch"])
print("Target labels accessed: False")
print("Training or checkpoint changed: False")

TRAINING HISTORY — SOURCE INFORMATION ONLY
epoch | cls_loss | mmd_loss | weighted_total | grad_norm | photo_F1 | art_F1 | cartoon_F1 | mean_F1
    1 |   1.8832 |   0.1906 |         2.0738 |   45.3876 |   0.0410 | 0.0359 |     0.0314 |  0.0361
    2 |   1.9372 |   0.1757 |         2.1128 |  112.3243 |   0.0585 | 0.0514 |     0.0421 |  0.0507
    3 |   1.9314 |   0.1718 |         2.1032 |  123.1494 |   0.0585 | 0.0514 |     0.0421 |  0.0507
    4 |   1.9260 |   0.2012 |         2.1272 |  159.9721 |   0.0585 | 0.0514 |     0.0421 |  0.0507
    5 |   1.9206 |   0.1887 |         2.1093 |   28.5433 |   0.0585 | 0.0514 |     0.0421 |  0.0507
    6 |   1.9164 |   0.1520 |         2.0685 |   32.1671 |   0.0585 | 0.0514 |     0.0421 |  0.0507
    7 |   1.9249 |   0.1785 |         2.1034 |  130.9124 |   0.0585 | 0.0514 |     0.0421 |  0.0507

SELECTED-CHECKPOINT SOURCE PREDICTION COUNTS

photo
Prediction counts: {6: 334}
Per-class source-validation accuracy: {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.

In [ ]:
# CELL 8 — measure classification versus MMD gradient scale.
# No training, no checkpoint changes, and no target labels.

import csv
import math
from pathlib import Path

import torch
import torch.nn.functional as F

from shared.mmd import three_kernel_mmd
from shared.pacs import (
    SEED,
    SOURCES,
    load_protocol,
    make_source_train_loader,
    make_target_train_loader,
    steps_per_source_epoch,
)
from task2.model import PACSClassifier, freeze_batchnorm_statistics
from task2.train import seed_everything

CODE_ROOT = Path("/content/task2-corrected-code")
RUN_ROOT = Path("/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
PROTOCOL_PATH = (
    CODE_ROOT / "shared" / "splits" / "pacs_sketch_seed6304.json"
)
INITIALIZATION_PATH = (
    RUN_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)

print("RECORDED EPOCH-1 COMPARISON")

for run_id in ("source_only", "dan_0p1", "dan_1"):
    history_path = RUN_ROOT / "runs" / run_id / "history.csv"

    with history_path.open(newline="") as handle:
        first_row = next(csv.DictReader(handle))

    print(
        run_id,
        {
            "classification_loss": round(
                float(first_row["classification_loss"]), 6
            ),
            "mmd_loss": round(float(first_row["mmd_loss"]), 6),
            "mmd_median": round(
                float(first_row["mmd_median_squared_distance"]), 6
            ),
            "gradient_norm": round(
                float(first_row["gradient_norm"]), 6
            ),
            "mean_source_macro_f1": round(
                float(first_row["mean_source_macro_f1"]), 6
            ),
        },
    )

device = torch.device("cuda")
protocol = load_protocol(PROTOCOL_PATH)
steps = steps_per_source_epoch(protocol)

initialization = torch.load(
    INITIALIZATION_PATH,
    map_location="cpu",
    weights_only=False,
)

seed_everything(SEED)

model = PACSClassifier(pretrained=False)
model.load_state_dict(initialization["state_dict"])
model.to(device)
model.train()
freeze_batchnorm_statistics(model)

source_loaders = {}

for domain_index, domain in enumerate(SOURCES):
    base_seed = SEED + 100_000 * (domain_index + 1)

    source_loaders[domain] = make_source_train_loader(
        PACS_ROOT,
        protocol["source_splits"][domain]["train"],
        steps,
        sampler_seed=base_seed,
        worker_seed=base_seed + 1,
        num_workers=2,
        pin_memory=True,
    )

target_seed = SEED + 900_000

target_loader = make_target_train_loader(
    PACS_ROOT,
    protocol["target_unlabeled"],
    steps,
    sampler_seed=target_seed,
    worker_seed=target_seed + 1,
    num_workers=2,
    pin_memory=True,
)

source_batches = [
    next(iter(source_loaders[domain]))
    for domain in SOURCES
]

source_images = torch.cat(
    [batch[0] for batch in source_batches],
    dim=0,
).to(device)

source_labels = torch.cat(
    [batch[1] for batch in source_batches],
    dim=0,
).to(device)

target_images, _ = next(iter(target_loader))
target_images = target_images.to(device)

combined_images = torch.cat(
    (source_images, target_images),
    dim=0,
)

logits, features = model(combined_images)

classification_loss = F.cross_entropy(
    logits[:24],
    source_labels,
    reduction="mean",
)

mmd_loss, diagnostics = three_kernel_mmd(
    features[:24],
    features[24:],
)

parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

classification_gradients = torch.autograd.grad(
    classification_loss,
    parameters,
    retain_graph=True,
    allow_unused=True,
)

mmd_gradients = torch.autograd.grad(
    mmd_loss,
    parameters,
    retain_graph=False,
    allow_unused=True,
)

def squared_norm(gradients):
    return sum(
        float(gradient.detach().float().square().sum().item())
        for gradient in gradients
        if gradient is not None
    )

classification_norm = math.sqrt(
    squared_norm(classification_gradients)
)
mmd_norm = math.sqrt(squared_norm(mmd_gradients))

dot_product = sum(
    float(
        (classification_gradient.detach().float()
         * mmd_gradient.detach().float()).sum().item()
    )
    for classification_gradient, mmd_gradient
    in zip(classification_gradients, mmd_gradients)
    if classification_gradient is not None
    and mmd_gradient is not None
)

cosine = dot_product / (
    classification_norm * mmd_norm
)

print("\nCOMMON-INITIALIZATION FIRST-BATCH DIAGNOSTIC")
print("Classification loss:", float(classification_loss.item()))
print("MMD loss:", float(mmd_loss.item()))
print(
    "MMD median squared distance:",
    diagnostics.median_squared_distance,
)
print("Classification gradient norm:", classification_norm)
print("Unweighted MMD gradient norm:", mmd_norm)
print(
    "MMD/classification gradient ratio:",
    mmd_norm / classification_norm,
)
print("Gradient cosine:", cosine)

for strength in (0.1, 1.0, 10.0):
    total_squared = 0.0

    for classification_gradient, mmd_gradient in zip(
        classification_gradients,
        mmd_gradients,
    ):
        if classification_gradient is None:
            combined_gradient = strength * mmd_gradient
        elif mmd_gradient is None:
            combined_gradient = classification_gradient
        else:
            combined_gradient = (
                classification_gradient
                + strength * mmd_gradient
            )

        if combined_gradient is not None:
            total_squared += float(
                combined_gradient.detach()
                .float()
                .square()
                .sum()
                .item()
            )

    print(
        f"Combined gradient norm at lambda={strength}:",
        math.sqrt(total_squared),
    )

print("\nTarget labels accessed: False")
print("Optimizer steps performed: 0")
print("Checkpoints changed: False")

RECORDED EPOCH-1 COMPARISON
source_only {'classification_loss': 0.470898, 'mmd_loss': 0.0, 'mmd_median': 0.0, 'gradient_norm': 11.497497, 'mean_source_macro_f1': 0.882794}
dan_0p1 {'classification_loss': 0.468411, 'mmd_loss': 0.264423, 'mmd_median': 1178.638882, 'gradient_norm': 10.908269, 'mean_source_macro_f1': 0.891465}
dan_1 {'classification_loss': 1.883205, 'mmd_loss': 0.190615, 'mmd_median': 6.871796, 'gradient_norm': 45.387631, 'mean_source_macro_f1': 0.036138}

COMMON-INITIALIZATION FIRST-BATCH DIAGNOSTIC
Classification loss: 1.9641157388687134
MMD loss: 0.7520227432250977
MMD median squared distance: 558.761474609375
Classification gradient norm: 12.717757451270664
Unweighted MMD gradient norm: 8.618231347567034
MMD/classification gradient ratio: 0.6776533819416382
Gradient cosine: 0.046623445932765066
Combined gradient norm at lambda=0.1: 12.786951260024692
Combined gradient norm at lambda=1.0: 15.69189455605853
Combined gradient norm at lambda=10.0: 87.70025843922858

Target

In [ ]:
# CELL 10 — install and verify the clipped v2 protocol.
# No training occurs in this cell.

import os
import sys
import json
import shutil
import hashlib
import zipfile
import platform
import subprocess
from pathlib import Path

import torch
import torchvision
import numpy as np
import sklearn

from google.colab import files

ARCHIVE_NAME = "ATML-PA1-Task2-corrected-clipped-v2.zip"
EXPECTED_ARCHIVE_SHA256 = (
    "dbd9174112bc1b8b20fa56d7c534295a1f2a5822fc1d0724ce4348f99f24ce6c"
)
EXPECTED_PREREGISTRATION_SHA256 = (
    "7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7"
)

OLD_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923"
)
NEW_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_clipped_v2_20260923"
)

CODE_ROOT = Path("/content/task2-corrected-code")
TEMPORARY_EXTRACTION = Path("/content/task2_v2_extraction")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError(f"Upload exactly one file: {ARCHIVE_NAME}")

uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
uploaded_hash = hashlib.sha256(uploaded_bytes).hexdigest()

if uploaded_hash != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Wrong or modified v2 archive.\n"
        f"Uploaded: {uploaded_name}\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {uploaded_hash}"
    )

local_archive = Path("/content") / ARCHIVE_NAME
local_archive.write_bytes(uploaded_bytes)

if TEMPORARY_EXTRACTION.exists():
    shutil.rmtree(TEMPORARY_EXTRACTION)
TEMPORARY_EXTRACTION.mkdir(parents=True)

with zipfile.ZipFile(local_archive) as archive:
    extraction_base = TEMPORARY_EXTRACTION.resolve()

    for member in archive.infolist():
        destination = (TEMPORARY_EXTRACTION / member.filename).resolve()
        if (
            destination != extraction_base
            and extraction_base not in destination.parents
        ):
            raise RuntimeError(f"Unsafe archive path: {member.filename}")

    archive.extractall(TEMPORARY_EXTRACTION)

extracted_root = TEMPORARY_EXTRACTION / "task2-corrected-code"
manifest_path = extracted_root / "task2" / "PACKAGE-MANIFEST.json"
manifest = json.loads(manifest_path.read_text())

if manifest["version"] != "2":
    raise RuntimeError("The uploaded package is not protocol version 2.")

if manifest["official_gradient_clipping"] != {
    "norm_type": 2.0,
    "max_norm": 20.0,
    "applies_to": "all_six_task2_configurations",
}:
    raise RuntimeError("The v2 clipping policy is incorrect.")

for record in manifest["files"]:
    path = extracted_root / record["path"]

    if not path.is_file():
        raise RuntimeError(f"Missing package file: {record['path']}")

    if sha256_file(path) != record["sha256"]:
        raise RuntimeError(f"Modified package file: {record['path']}")

preregistration_in_package = (
    extracted_root
    / "task2"
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)

if (
    sha256_file(preregistration_in_package)
    != EXPECTED_PREREGISTRATION_SHA256
):
    raise RuntimeError("Locked expectation hash is incorrect.")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

shutil.move(str(extracted_root), str(CODE_ROOT))
shutil.rmtree(TEMPORARY_EXTRACTION)

NEW_ROOT.mkdir(parents=True, exist_ok=True)

saved_archive = NEW_ROOT / "code" / ARCHIVE_NAME
saved_archive.parent.mkdir(parents=True, exist_ok=True)

if saved_archive.exists():
    if sha256_file(saved_archive) != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError("A different v2 archive exists in the new root.")
else:
    saved_archive.write_bytes(uploaded_bytes)

new_preregistration = (
    NEW_ROOT
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)
new_preregistration.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(preregistration_in_package, new_preregistration)

old_initialization = (
    OLD_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)
new_initialization = (
    NEW_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)
new_initialization.parent.mkdir(parents=True, exist_ok=True)

if not old_initialization.is_file():
    raise FileNotFoundError(
        f"Verified common initialization is missing: {old_initialization}"
    )

if new_initialization.exists():
    if sha256_file(new_initialization) != sha256_file(old_initialization):
        raise RuntimeError("The v2 initialization differs from the verified original.")
else:
    shutil.copy2(old_initialization, new_initialization)

current_environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
    "numpy": str(np.__version__),
    "sklearn": str(sklearn.__version__),
    "cuda": str(torch.version.cuda),
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
}

old_environment = json.loads(
    (OLD_ROOT / "environment_preflight.json").read_text()
)

for key, value in current_environment.items():
    if old_environment.get(key) != value:
        raise RuntimeError(
            f"Runtime changed at {key}: "
            f"{old_environment.get(key)!r} != {value!r}"
        )

v2_environment = {
    **current_environment,
    "package_sha256": EXPECTED_ARCHIVE_SHA256,
    "gradient_clipping_max_norm": 20.0,
}

new_environment_path = NEW_ROOT / "environment_preflight.json"

if new_environment_path.exists():
    if json.loads(new_environment_path.read_text()) != v2_environment:
        raise RuntimeError("Existing v2 environment record differs.")
else:
    new_environment_path.write_text(
        json.dumps(v2_environment, indent=2) + "\n"
    )

environment = os.environ.copy()
environment["PYTHONPATH"] = str(CODE_ROOT)
environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

test_program = """
from tests import test_locked_choices
from task2.config import RUNS, load_config

tests_run = 0

for name in sorted(dir(test_locked_choices)):
    if name.startswith("test_"):
        getattr(test_locked_choices, name)()
        tests_run += 1

for run_id in RUNS:
    config = load_config(run_id)
    assert config["gradient_clipping"] == 20.0
    assert config["gradient_clipping_norm_type"] == 2.0
    assert config["gradient_clipping_foreach"] is False

print(f"V2 TESTS PASSED: {tests_run}")
print(f"CLIPPED CONFIGURATIONS VERIFIED: {len(RUNS)}")
"""

subprocess.run(
    [sys.executable, "-c", test_program],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

preflight_output = NEW_ROOT / "preflight" / "data_preflight.json"
preflight_output.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared"
            / "splits"
            / "pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_output),
    ],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

preflight = json.loads(preflight_output.read_text())

print("\nCLIPPED V2 INSTALLED AND VERIFIED")
print("Package SHA256:", uploaded_hash)
print("Code files verified:", len(manifest["files"]))
print("Expectation SHA256:", sha256_file(new_preregistration))
print("Initialization SHA256:", sha256_file(new_initialization))
print("Preflight:", preflight["status"])
print("Official gradient max norm:", 20.0)
print("Training started:", False)
print("Target labels accessed:", False)
print("New official root:", NEW_ROOT)

Saving ATML-PA1-Task2-corrected-clipped-v2.zip to ATML-PA1-Task2-corrected-clipped-v2.zip


FileNotFoundError: [Errno 2] No such file or directory: '/content/task2_v2_extraction/task2-corrected-code/task2/preregistration/DAN_STRENGTH_EXPECTATION.txt'

In [ ]:
# CELL 10B — continue safely from the interrupted v2 installation.

import os
import sys
import json
import shutil
import hashlib
import platform
import subprocess
from pathlib import Path

import torch
import torchvision
import numpy as np
import sklearn

EXPECTED_ARCHIVE_SHA256 = (
    "dbd9174112bc1b8b20fa56d7c534295a1f2a5822fc1d0724ce4348f99f24ce6c"
)
EXPECTED_PREREGISTRATION_SHA256 = (
    "7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7"
)

CODE_ROOT = Path("/content/task2-corrected-code")
OLD_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_20260923"
)
NEW_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_clipped_v2_20260923"
)
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest_path = CODE_ROOT / "task2" / "PACKAGE-MANIFEST.json"

if not manifest_path.is_file():
    raise FileNotFoundError("The installed v2 manifest is missing.")

manifest = json.loads(manifest_path.read_text())

if manifest["version"] != "2":
    raise RuntimeError("Installed code is not protocol v2.")

for record in manifest["files"]:
    path = CODE_ROOT / record["path"]

    if not path.is_file():
        raise RuntimeError(f"Installed file missing: {record['path']}")

    if sha256_file(path) != record["sha256"]:
        raise RuntimeError(f"Installed file changed: {record['path']}")

package_preregistration = (
    CODE_ROOT
    / "task2"
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)
new_preregistration = (
    NEW_ROOT
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)

if (
    sha256_file(package_preregistration)
    != EXPECTED_PREREGISTRATION_SHA256
):
    raise RuntimeError("Installed expectation hash is incorrect.")

new_preregistration.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(package_preregistration, new_preregistration)

old_initialization = (
    OLD_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)
new_initialization = (
    NEW_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)

if not old_initialization.is_file():
    raise FileNotFoundError("The verified common initialization is missing.")

new_initialization.parent.mkdir(parents=True, exist_ok=True)

if new_initialization.exists():
    if sha256_file(new_initialization) != sha256_file(old_initialization):
        raise RuntimeError("Existing v2 initialization differs.")
else:
    shutil.copy2(old_initialization, new_initialization)

saved_archive = (
    NEW_ROOT
    / "code"
    / "ATML-PA1-Task2-corrected-clipped-v2.zip"
)

if not saved_archive.is_file():
    raise FileNotFoundError("The verified v2 archive was not saved.")

if sha256_file(saved_archive) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("Saved v2 archive hash is incorrect.")

current_environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
    "numpy": str(np.__version__),
    "sklearn": str(sklearn.__version__),
    "cuda": str(torch.version.cuda),
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
}

old_environment = json.loads(
    (OLD_ROOT / "environment_preflight.json").read_text()
)

for key, value in current_environment.items():
    if old_environment.get(key) != value:
        raise RuntimeError(
            f"Runtime changed at {key}: "
            f"{old_environment.get(key)!r} != {value!r}"
        )

v2_environment = {
    **current_environment,
    "package_sha256": EXPECTED_ARCHIVE_SHA256,
    "gradient_clipping_max_norm": 20.0,
}

new_environment_path = NEW_ROOT / "environment_preflight.json"

if new_environment_path.exists():
    if json.loads(new_environment_path.read_text()) != v2_environment:
        raise RuntimeError("Existing v2 environment record differs.")
else:
    new_environment_path.write_text(
        json.dumps(v2_environment, indent=2) + "\n"
    )

environment = os.environ.copy()
environment["PYTHONPATH"] = str(CODE_ROOT)
environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

test_program = """
from tests import test_locked_choices
from task2.config import RUNS, load_config

tests_run = 0

for name in sorted(dir(test_locked_choices)):
    if name.startswith("test_"):
        getattr(test_locked_choices, name)()
        tests_run += 1

for run_id in RUNS:
    config = load_config(run_id)
    assert config["gradient_clipping"] == 20.0
    assert config["gradient_clipping_norm_type"] == 2.0
    assert config["gradient_clipping_foreach"] is False

print(f"V2 TESTS PASSED: {tests_run}")
print(f"CLIPPED CONFIGURATIONS VERIFIED: {len(RUNS)}")
"""

subprocess.run(
    [sys.executable, "-c", test_program],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

preflight_output = NEW_ROOT / "preflight" / "data_preflight.json"
preflight_output.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared"
            / "splits"
            / "pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_output),
    ],
    cwd=CODE_ROOT,
    env=environment,
    check=True,
)

preflight = json.loads(preflight_output.read_text())

print("\nCLIPPED V2 RECOVERY COMPLETE")
print("Code files verified:", len(manifest["files"]))
print("Package SHA256:", sha256_file(saved_archive))
print("Expectation SHA256:", sha256_file(new_preregistration))
print("Initialization SHA256:", sha256_file(new_initialization))
print("Preflight:", preflight["status"])
print("Official gradient max norm:", 20.0)
print("Training started:", False)
print("Target labels accessed:", False)
print("New official root:", NEW_ROOT)

CalledProcessError: Command '['/usr/bin/python3', '-c', '\nfrom tests import test_locked_choices\nfrom task2.config import RUNS, load_config\n\ntests_run = 0\n\nfor name in sorted(dir(test_locked_choices)):\n    if name.startswith("test_"):\n        getattr(test_locked_choices, name)()\n        tests_run += 1\n\nfor run_id in RUNS:\n    config = load_config(run_id)\n    assert config["gradient_clipping"] == 20.0\n    assert config["gradient_clipping_norm_type"] == 2.0\n    assert config["gradient_clipping_foreach"] is False\n\nprint(f"V2 TESTS PASSED: {tests_run}")\nprint(f"CLIPPED CONFIGURATIONS VERIFIED: {len(RUNS)}")\n']' returned non-zero exit status 1.

In [ ]:
# CELL 10C — run v2 tests using the exact file path, then finish preflight.

import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path

CODE_ROOT = Path("/content/task2-corrected-code")
NEW_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_clipped_v2_20260923"
)
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

EXPECTED_ARCHIVE_SHA256 = (
    "dbd9174112bc1b8b20fa56d7c534295a1f2a5822fc1d0724ce4348f99f24ce6c"
)
EXPECTED_PREREGISTRATION_SHA256 = (
    "7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

environment = os.environ.copy()
environment["PYTHONPATH"] = str(CODE_ROOT)
environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

test_file = CODE_ROOT / "tests" / "test_locked_choices.py"

test_program = f"""
import importlib.util
from pathlib import Path
from task2.config import RUNS, load_config

test_path = Path({str(test_file)!r})
spec = importlib.util.spec_from_file_location(
    "task2_v2_locked_tests",
    test_path,
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

tests_run = 0

for name in sorted(dir(module)):
    if name.startswith("test_"):
        getattr(module, name)()
        tests_run += 1

for run_id in RUNS:
    config = load_config(run_id)
    assert config["gradient_clipping"] == 20.0
    assert config["gradient_clipping_norm_type"] == 2.0
    assert config["gradient_clipping_foreach"] is False

print(f"V2 TESTS PASSED: {{tests_run}}")
print(f"CLIPPED CONFIGURATIONS VERIFIED: {{len(RUNS)}}")
"""

test_result = subprocess.run(
    [sys.executable, "-c", test_program],
    cwd=CODE_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)

print(test_result.stdout)

if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("The exact-path v2 tests failed; output shown above.")

preflight_output = NEW_ROOT / "preflight" / "data_preflight.json"
preflight_output.parent.mkdir(parents=True, exist_ok=True)

preflight_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared"
            / "splits"
            / "pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_output),
    ],
    cwd=CODE_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)

print(preflight_result.stdout)

if preflight_result.returncode != 0:
    print(preflight_result.stderr)
    raise RuntimeError("The v2 data preflight failed; output shown above.")

preflight = json.loads(preflight_output.read_text())

saved_archive = (
    NEW_ROOT
    / "code"
    / "ATML-PA1-Task2-corrected-clipped-v2.zip"
)
preregistration = (
    NEW_ROOT
    / "preregistration"
    / "DAN_STRENGTH_EXPECTATION.txt"
)
initialization = (
    NEW_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)

if sha256_file(saved_archive) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("Saved v2 archive hash mismatch.")

if sha256_file(preregistration) != EXPECTED_PREREGISTRATION_SHA256:
    raise RuntimeError("Expectation hash mismatch.")

print("\nCLIPPED V2 RECOVERY COMPLETE")
print("Package SHA256:", sha256_file(saved_archive))
print("Expectation SHA256:", sha256_file(preregistration))
print("Initialization SHA256:", sha256_file(initialization))
print("Preflight:", preflight["status"])
print("Official gradient max norm:", 20.0)
print("Training started:", False)
print("Target labels accessed:", False)
print("New official root:", NEW_ROOT)

V2 TESTS PASSED: 7
CLIPPED CONFIGURATIONS VERIFIED: 6

{
  "status": "PREFLIGHT_PASS",
  "training_started": false,
  "target_labels_accessed": false,
  "dataset": {
    "root": "/content/atml_pacs/pacs/images",
    "file_list_sha256": "559ac63b8df8e07330b97112e5ec4c414b3957585b28d21b4cfecd2181f538e0",
    "photo": 1670,
    "art_painting": 2048,
    "cartoon": 2344,
    "sketch": 3929
  },
  "source_counts": {
    "photo": {
      "train": 1336,
      "validation": 334
    },
    "art_painting": {
      "train": 1638,
      "validation": 410
    },
    "cartoon": {
      "train": 1875,
      "validation": 469
    }
  },
  "target_unlabeled_count": 3929,
  "steps_per_source_epoch": 235,
  "planned_updates": 7050,
  "registered_runs": [
    "source_only",
    "dan_0p1",
    "dan_1",
    "dan_10",
    "dann",
    "cdan"
  ],
  "environment": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "numpy": "2.1.3",
    "sklearn": "1.6.1",
    "cuda":

In [ ]:
import importlib
import sys
from pathlib import Path

CODE_ROOT = Path("/content/task2-corrected-code")

# Remove cached Task 2 modules from the earlier notebook execution.
for module_name in list(sys.modules):
    if (
        module_name == "task2"
        or module_name.startswith("task2.")
        or module_name == "shared"
        or module_name.startswith("shared.")
    ):
        del sys.modules[module_name]

sys.path = [
    str(CODE_ROOT),
    *[path for path in sys.path if path != str(CODE_ROOT)],
]
importlib.invalidate_caches()

from task2.config import load_config

config = load_config("dan_1")
assert config["gradient_clipping"] == 20.0

print("Fresh v2 module loaded from:", sys.modules["task2.config"].__file__)
print("Gradient clipping:", config["gradient_clipping"])
print("TRAINING STARTED: False")

Fresh v2 module loaded from: /content/task2-corrected-code/task2/config.py
Gradient clipping: 20.0
TRAINING STARTED: False


In [ ]:
# OFFICIAL CLIPPED V2 — DAN lambda_MMD=1 stability gate
# No target labels are accessed. No settings are changed here.

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

import torch

CODE_ROOT = Path("/content/task2-corrected-code")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_clipped_v2_20260923"
)

PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
RUN_DIR = OUTPUT_ROOT / "dan_1"

EXPECTED_INITIALIZATION_SHA256 = (
    "567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ------------------------------------------------------------------
# Pre-run safeguards
# ------------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.get_device_name(0) == "Tesla T4", (
    f"Expected Tesla T4, found {torch.cuda.get_device_name(0)}"
)

for required in (
    CODE_ROOT,
    PACS_ROOT,
    PROTOCOL,
    PREREGISTRATION,
    INITIALIZATION,
):
    assert required.exists(), f"Missing required path: {required}"

assert sha256_file(INITIALIZATION) == EXPECTED_INITIALIZATION_SHA256, (
    "The common initialization file has changed."
)

sys.path.insert(0, str(CODE_ROOT))

from task2.config import load_config

config = load_config("dan_1")

assert config["method"] == "dan"
assert config["mmd_lambda"] == 1.0
assert config["gradient_clipping"] == 20.0
assert config["gradient_clipping_norm_type"] == 2.0
assert config["gradient_clipping_foreach"] is False

if RUN_DIR.exists():
    raise RuntimeError(
        f"{RUN_DIR} already exists. Do not delete or overwrite it. "
        "Share its contents before deciding whether to resume."
    )

print("Starting official clipped DAN lambda_MMD=1.")
print("GPU:", torch.cuda.get_device_name(0))
print("Gradient clipping: global L2 max_norm=20")
print("Target labels accessed: False")

# ------------------------------------------------------------------
# Train in a fresh Python subprocess
# ------------------------------------------------------------------

command = [
    sys.executable,
    "-m",
    "task2.train",
    "--run-id",
    "dan_1",
    "--pacs-root",
    str(PACS_ROOT),
    "--dataset-source",
    (
        "Google Drive PACS_dassl.zip; "
        "sha256=0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
    ),
    "--protocol",
    str(PROTOCOL),
    "--initialization",
    str(INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--output",
    str(OUTPUT_ROOT),
    "--device",
    "cuda",
    "--num-workers",
    "2",
]

subprocess.run(command, cwd=CODE_ROOT, check=True)

# ------------------------------------------------------------------
# Verify saved run without retraining or target-label evaluation
# ------------------------------------------------------------------

required_outputs = [
    RUN_DIR / "best.pt",
    RUN_DIR / "last.pt",
    RUN_DIR / "history.csv",
    RUN_DIR / "best_source_validation.json",
    RUN_DIR / "run.json",
    OUTPUT_ROOT / "experiment_lock.json",
]

for required in required_outputs:
    assert required.is_file(), f"Missing run output: {required}"

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

assert history, "Training history is empty."
assert run_record["run_id"] == "dan_1"
assert run_record["target_labels_used"] is False
assert run_record["identity"]["config"]["mmd_lambda"] == 1.0
assert run_record["identity"]["config"]["gradient_clipping"] == 20.0

numeric_columns = [
    name for name in history[0]
    if name != "epoch"
]

for row in history:
    for name in numeric_columns:
        value = float(row[name])
        assert math.isfinite(value), (
            f"Non-finite history value: epoch={row['epoch']}, "
            f"column={name}, value={value}"
        )

post_clip_norms = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
clipped_fractions = [
    float(row["gradient_clipped_fraction"])
    for row in history
]

assert max(post_clip_norms) <= 20.0002
assert all(0.0 <= value <= 1.0 for value in clipped_fractions)

source_f1_values = [
    float(row["mean_source_macro_f1"])
    for row in history
]
expected_best_index = max(
    range(len(source_f1_values)),
    key=lambda index: source_f1_values[index],
)
expected_best_epoch = int(history[expected_best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert math.isclose(
    run_record["best_mean_source_macro_f1"],
    source_f1_values[expected_best_index],
    rel_tol=0.0,
    abs_tol=1e-12,
)
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

best_checkpoint = torch.load(
    RUN_DIR / "best.pt",
    map_location="cpu",
    weights_only=False,
)
last_checkpoint = torch.load(
    RUN_DIR / "last.pt",
    map_location="cpu",
    weights_only=False,
)
initialization = torch.load(
    INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)

assert best_checkpoint["target_labels_used"] is False
assert last_checkpoint["target_labels_used"] is False

# Confirm the assignment's frozen BatchNorm-buffer policy.
bn_suffixes = (
    "running_mean",
    "running_var",
    "num_batches_tracked",
)

for name, initial_value in initialization["state_dict"].items():
    if name.endswith(bn_suffixes):
        assert torch.equal(
            initial_value,
            best_checkpoint["model_state"][name],
        ), f"BatchNorm buffer changed: {name}"

print("\nOFFICIAL CLIPPED DAN lambda_MMD=1 VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Maximum recorded post-clipping gradient norm:",
    max(post_clip_norms),
)
print(
    "Mean fraction of updates clipped:",
    sum(clipped_fractions) / len(clipped_fractions),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("BatchNorm buffers unchanged: True")
print("Target labels accessed: False")

Starting official clipped DAN lambda_MMD=1.
GPU: Tesla T4
Gradient clipping: global L2 max_norm=20
Target labels accessed: False

OFFICIAL CLIPPED DAN lambda_MMD=1 VERIFIED COMPLETE
Epochs completed: 6
Selected epoch: 1
Best mean source-validation macro-F1: 0.1161215679938203
Maximum recorded post-clipping gradient norm: 19.89729237374047
Mean fraction of updates clipped: 0.8673758865248228
Checkpoint SHA256: 572f2c40d9498e779e2e56ab4de4fb882b13d264eea97e838a42939b829f9761
BatchNorm buffers unchanged: True
Target labels accessed: False


In [ ]:
# READ-ONLY DIAGNOSTIC FOR CLIPPED DAN lambda_MMD=1
# Does not train, alter checkpoints, or access Sketch labels.

from pathlib import Path
from collections import Counter
import csv
import importlib
import json
import sys

import torch
from sklearn.metrics import accuracy_score, f1_score

CODE_ROOT = Path("/content/task2-corrected-code")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/task2_corrected_clipped_v2_20260923"
)
RUN_DIR = OUTPUT_ROOT / "dan_1"
PROTOCOL_PATH = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"

# Ensure the current v2 modules are used.
for module_name in list(sys.modules):
    if (
        module_name == "task2"
        or module_name.startswith("task2.")
        or module_name == "shared"
        or module_name.startswith("shared.")
    ):
        del sys.modules[module_name]

sys.path = [
    str(CODE_ROOT),
    *[path for path in sys.path if path != str(CODE_ROOT)],
]
importlib.invalidate_caches()

from shared.pacs import SOURCES, load_protocol, make_validation_loader
from task2.model import PACSClassifier

# ------------------------------------------------------------------
# Training history
# ------------------------------------------------------------------

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

print("CLIPPED DAN lambda_MMD=1 — SOURCE-ONLY DIAGNOSTIC")
print()
print(
    "epoch | cls_loss | mmd_loss | median_d2 | "
    "grad_before | grad_after | clipped_frac | mean_source_F1"
)

for row in history:
    print(
        f"{int(row['epoch']):5d} | "
        f"{float(row['classification_loss']):8.4f} | "
        f"{float(row['mmd_loss']):8.4f} | "
        f"{float(row['mmd_median_squared_distance']):9.4f} | "
        f"{float(row['gradient_norm']):11.4f} | "
        f"{float(row['gradient_norm_after_clipping']):10.4f} | "
        f"{float(row['gradient_clipped_fraction']):12.4f} | "
        f"{float(row['mean_source_macro_f1']):14.4f}"
    )

# ------------------------------------------------------------------
# Re-evaluate the frozen selected checkpoint on source validation only
# ------------------------------------------------------------------

protocol = load_protocol(PROTOCOL_PATH)

checkpoint = torch.load(
    RUN_DIR / "best.pt",
    map_location="cpu",
    weights_only=False,
)
assert checkpoint["target_labels_used"] is False

device = torch.device("cuda")
model = PACSClassifier(pretrained=False)
model.load_state_dict(checkpoint["model_state"])
model.to(device)
model.eval()

saved_metrics = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

all_predictions = Counter()

print()
print("SELECTED CHECKPOINT:", checkpoint["epoch"])

for domain in SOURCES:
    loader = make_validation_loader(
        PACS_ROOT,
        protocol["source_splits"][domain]["validation"],
        batch_size=64,
        num_workers=2,
        pin_memory=True,
    )

    truth = []
    predictions = []

    with torch.inference_mode():
        for images, labels, _ in loader:
            logits, _ = model(images.to(device, non_blocking=True))
            predicted = logits.argmax(dim=1).cpu().tolist()

            truth.extend(int(value) for value in labels.tolist())
            predictions.extend(int(value) for value in predicted)

    counts = Counter(predictions)
    all_predictions.update(predictions)

    accuracy = accuracy_score(truth, predictions)
    macro_f1 = f1_score(
        truth,
        predictions,
        labels=list(range(7)),
        average="macro",
        zero_division=0,
    )

    assert abs(
        accuracy - saved_metrics[f"{domain}_accuracy"]
    ) < 1e-12
    assert abs(
        macro_f1 - saved_metrics[f"{domain}_macro_f1"]
    ) < 1e-12

    print()
    print(domain)
    print("  prediction counts:", dict(sorted(counts.items())))
    print("  accuracy:", round(accuracy, 6))
    print("  macro-F1:", round(macro_f1, 6))

print()
print("POOLED SOURCE PREDICTION COUNTS:", dict(sorted(all_predictions.items())))
print("Checkpoint unchanged: True")
print("Training performed: False")
print("Target labels accessed: False")

CLIPPED DAN lambda_MMD=1 — SOURCE-ONLY DIAGNOSTIC

epoch | cls_loss | mmd_loss | median_d2 | grad_before | grad_after | clipped_frac | mean_source_F1
    1 |   1.7697 |   0.1847 |    6.8731 |     24.4219 |    14.9911 |       0.4170 |         0.1161
    2 |   1.7924 |   0.2093 |    0.0002 |    111.4720 |    19.6522 |       0.9319 |         0.0985
    3 |   1.7749 |   0.1953 |    0.0000 |    112.6535 |    19.8594 |       0.9660 |         0.0892
    4 |   1.8082 |   0.1772 |    0.0000 |    162.7841 |    19.8725 |       0.9787 |         0.0361
    5 |   1.9353 |   0.2014 |    0.0000 |    181.0196 |    19.8973 |       0.9745 |         0.0364
    6 |   1.9270 |   0.1949 |    0.0000 |    107.8558 |    19.7382 |       0.9362 |         0.0507

SELECTED CHECKPOINT: 1

photo
  prediction counts: {5: 246, 6: 88}
  accuracy: 0.422156
  macro-F1: 0.192553

art_painting
  prediction counts: {5: 375, 6: 35}
  accuracy: 0.204878
  macro-F1: 0.095984

cartoon
  prediction counts: {5: 457, 6: 12}
  accur

In [ ]:
# INSTALL AND PREFLIGHT TASK 2 NORMALIZED-MMD V3
# This cell performs no training and accesses no target labels.

from google.colab import files, drive
from pathlib import Path

import hashlib
import json
import shutil
import subprocess
import sys
import tempfile
import zipfile

drive.mount("/content/drive")

PACKAGE_NAME = "ATML-PA1-Task2-corrected-normalized-v3.zip"
EXPECTED_PACKAGE_SHA256 = (
    "52771bf601bc67522fb8bea6f59fb6ad6d855957cd8a6c99ef5cd08c574de787"
)
EXPECTED_EXPECTATION_SHA256 = (
    "7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7"
)
EXPECTED_INITIALIZATION_SHA256 = (
    "567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622"
)

PACS_ROOT = Path("/content/atml_pacs/pacs/images")
CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

V2_INITIALIZATION = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_clipped_v2_20260923/"
    "initialization/resnet18_v1_seed6304_common.pt"
)
V3_INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ------------------------------------------------------------------
# Upload and verify
# ------------------------------------------------------------------

uploaded = files.upload()
assert PACKAGE_NAME in uploaded, f"Please upload exactly {PACKAGE_NAME}"

package_path = Path("/content") / PACKAGE_NAME
assert sha256_file(package_path) == EXPECTED_PACKAGE_SHA256, (
    "The uploaded v3 archive has the wrong SHA256."
)

if CODE_ROOT.exists():
    raise RuntimeError(
        f"{CODE_ROOT} already exists. Do not overwrite it silently."
    )

temporary_root = Path(tempfile.mkdtemp(prefix="task2_v3_extract_"))

with zipfile.ZipFile(package_path) as archive:
    archive.extractall(temporary_root)

extracted_root = temporary_root / "task2-corrected-normalized-v3"
assert extracted_root.is_dir(), "Unexpected archive structure."

shutil.copytree(extracted_root, CODE_ROOT)

# ------------------------------------------------------------------
# Verify the governed repository-overlay manifest
# ------------------------------------------------------------------

manifest_path = CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
manifest = json.loads(manifest_path.read_text())

assert manifest["version"] == "3"
assert manifest["training_started"] is False
assert manifest["target_evaluation_included"] is False
assert (
    manifest["official_mmd_feature_normalization"]["method"]
    == "l2_per_sample"
)
assert manifest["official_gradient_clipping"]["max_norm"] == 20.0

for entry in manifest["files"]:
    path = CODE_ROOT / entry["path"]
    assert path.is_file(), f"Missing manifested file: {entry['path']}"
    assert path.stat().st_size == entry["bytes"]
    assert sha256_file(path) == entry["sha256"], (
        f"Manifest hash mismatch: {entry['path']}"
    )

preregistration = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
assert sha256_file(preregistration) == EXPECTED_EXPECTATION_SHA256

# ------------------------------------------------------------------
# Run all eight focused tests in a clean subprocess
# ------------------------------------------------------------------

test_program = r'''
import importlib.util
from pathlib import Path

from task2.config import RUNS, load_config

test_path = Path("tests/test_locked_choices.py")
spec = importlib.util.spec_from_file_location("task2_v3_tests", test_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

tests_run = 0
for name in sorted(dir(module)):
    if name.startswith("test_"):
        getattr(module, name)()
        tests_run += 1

assert tests_run == 8

for run_id in RUNS:
    config = load_config(run_id)
    assert config["gradient_clipping"] == 20.0
    assert config["mmd_feature_normalization"] == "l2_per_sample"

print(f"V3 TESTS PASSED: {tests_run}")
print(f"V3 CONFIGURATIONS VERIFIED: {len(RUNS)}")
'''

subprocess.run(
    [sys.executable, "-c", test_program],
    cwd=CODE_ROOT,
    check=True,
)

# ------------------------------------------------------------------
# Reuse and verify the exact common initialization
# ------------------------------------------------------------------

assert V2_INITIALIZATION.is_file(), (
    f"Missing verified initialization: {V2_INITIALIZATION}"
)
assert sha256_file(V2_INITIALIZATION) == EXPECTED_INITIALIZATION_SHA256

V3_INITIALIZATION.parent.mkdir(parents=True, exist_ok=True)

if V3_INITIALIZATION.exists():
    assert sha256_file(V3_INITIALIZATION) == EXPECTED_INITIALIZATION_SHA256
else:
    shutil.copy2(V2_INITIALIZATION, V3_INITIALIZATION)

assert sha256_file(V3_INITIALIZATION) == EXPECTED_INITIALIZATION_SHA256

# ------------------------------------------------------------------
# Dataset and configuration preflight
# ------------------------------------------------------------------

assert PACS_ROOT.is_dir(), f"Missing PACS data: {PACS_ROOT}"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
preflight_path = OUTPUT_ROOT / "preflight.json"

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"),
        "--output",
        str(preflight_path),
    ],
    cwd=CODE_ROOT,
    check=True,
)

preflight = json.loads(preflight_path.read_text())

assert preflight["status"] == "PREFLIGHT_PASS"
assert preflight["training_started"] is False
assert preflight["target_labels_accessed"] is False
assert preflight["environment"]["gpu"] == "Tesla T4"

print()
print("NORMALIZED-MMD V3 INSTALLATION VERIFIED")
print("Package SHA256:", sha256_file(package_path))
print("Governed files:", len(manifest["files"]))
print("Expectation SHA256:", sha256_file(preregistration))
print("Initialization SHA256:", sha256_file(V3_INITIALIZATION))
print("MMD normalization: per-example L2, DAN only")
print("Gradient clipping: global L2 max_norm=20")
print("Preflight:", preflight["status"])
print("Training started: False")
print("Target labels accessed: False")
print("Code root:", CODE_ROOT)
print("New v3 output root:", OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving ATML-PA1-Task2-corrected-normalized-v3.zip to ATML-PA1-Task2-corrected-normalized-v3.zip

NORMALIZED-MMD V3 INSTALLATION VERIFIED
Package SHA256: 52771bf601bc67522fb8bea6f59fb6ad6d855957cd8a6c99ef5cd08c574de787
Governed files: 26
Expectation SHA256: 7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7
Initialization SHA256: 567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622
MMD normalization: per-example L2, DAN only
Gradient clipping: global L2 max_norm=20
Preflight: PREFLIGHT_PASS
Training started: False
Target labels accessed: False
Code root: /content/task2-corrected-normalized-v3
New v3 output root: /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923


In [ ]:
# TASK 2 V3 — DAN lambda_MMD=1 STABILITY GATE
# Uses L2-normalized features only inside MMD.
# Does not evaluate or access Sketch labels.

from pathlib import Path
import hashlib
import json
import subprocess
import sys

import torch

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
PREFLIGHT = OUTPUT_ROOT / "preflight.json"
RUN_DIR = OUTPUT_ROOT / "dan_1"

GITHUB_RECORD = {
    "repository": (
        "https://github.com/"
        "therealshaheer11-glithc/ATML-Assignment-1"
    ),
    "commit": "b96f184",
    "protocol": "normalized-MMD-v3",
    "target_labels_accessed": False,
}

EXPECTED_INITIALIZATION_SHA256 = (
    "567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622"
)
EXPECTED_EXPECTATION_SHA256 = (
    "7cb1a3988b92c5986d850e65d3dc1a928af9d673d19b53b97dd1d90b78a31fb7"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ------------------------------------------------------------------
# Pre-run checks
# ------------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.get_device_name(0) == "Tesla T4"

for required in (
    CODE_ROOT,
    PACS_ROOT,
    PROTOCOL,
    PREREGISTRATION,
    INITIALIZATION,
    PREFLIGHT,
):
    assert required.exists(), f"Missing required path: {required}"

assert sha256_file(INITIALIZATION) == EXPECTED_INITIALIZATION_SHA256
assert sha256_file(PREREGISTRATION) == EXPECTED_EXPECTATION_SHA256

preflight = json.loads(PREFLIGHT.read_text())
assert preflight["status"] == "PREFLIGHT_PASS"
assert preflight["training_started"] is False
assert preflight["target_labels_accessed"] is False
assert preflight["environment"]["gpu"] == "Tesla T4"

if RUN_DIR.exists():
    raise RuntimeError(
        f"{RUN_DIR} already exists. Do not delete or overwrite it."
    )

github_record_path = OUTPUT_ROOT / "github_commit.json"

if github_record_path.exists():
    assert json.loads(github_record_path.read_text()) == GITHUB_RECORD
else:
    github_record_path.write_text(
        json.dumps(GITHUB_RECORD, indent=2) + "\n"
    )

# Verify the fresh subprocess sees v3 settings.
configuration_check = r'''
from task2.config import load_config

config = load_config("dan_1")
assert config["method"] == "dan"
assert config["mmd_lambda"] == 1.0
assert config["mmd_feature_normalization"] == "l2_per_sample"
assert config["gradient_clipping"] == 20.0

print("V3 configuration verified.")
'''

subprocess.run(
    [sys.executable, "-c", configuration_check],
    cwd=CODE_ROOT,
    check=True,
)

print("Starting normalized-MMD v3 DAN lambda_MMD=1.")
print("GitHub commit: b96f184")
print("GPU:", torch.cuda.get_device_name(0))
print("MMD feature treatment: per-example L2 normalization")
print("Classification feature treatment: unchanged")
print("Gradient clipping: global L2 max_norm=20")
print("Target labels accessed: False")

# ------------------------------------------------------------------
# Train in a fresh subprocess
# ------------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id",
        "dan_1",
        "--pacs-root",
        str(PACS_ROOT),
        "--dataset-source",
        (
            "Google Drive PACS_dassl.zip; "
            "sha256="
            "0dc9d0176fa27c9b4504e7c2e962aebe"
            "6a79ed0c1819b84148786e590f87e102"
        ),
        "--protocol",
        str(PROTOCOL),
        "--initialization",
        str(INITIALIZATION),
        "--preregistration",
        str(PREREGISTRATION),
        "--output",
        str(OUTPUT_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
    ],
    cwd=CODE_ROOT,
    check=True,
)

# ------------------------------------------------------------------
# Verify and diagnose using source validation only
# ------------------------------------------------------------------

verification_program = r'''
from collections import Counter
from pathlib import Path

import csv
import hashlib
import json
import math

import torch
from sklearn.metrics import accuracy_score, f1_score

from shared.pacs import (
    SOURCES,
    load_protocol,
    make_validation_loader,
)
from task2.config import load_config
from task2.model import PACSClassifier
from task2.train import hash_code_tree

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
RUN_DIR = OUTPUT_ROOT / "dan_1"
PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required = [
    RUN_DIR / "best.pt",
    RUN_DIR / "last.pt",
    RUN_DIR / "history.csv",
    RUN_DIR / "best_source_validation.json",
    RUN_DIR / "run.json",
    OUTPUT_ROOT / "experiment_lock.json",
    OUTPUT_ROOT / "github_commit.json",
]

for path in required:
    assert path.is_file(), f"Missing output: {path}"

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)
config = load_config("dan_1")

assert config["mmd_feature_normalization"] == "l2_per_sample"
assert config["gradient_clipping"] == 20.0
assert run_record["target_labels_used"] is False
assert run_record["identity"]["config"] == config
assert run_record["identity"]["code_sha256"] == hash_code_tree(CODE_ROOT)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

assert history

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value)), (
                f"Non-finite value at epoch {row['epoch']}: {name}"
            )

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
expected_best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[expected_best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert math.isclose(
    run_record["best_mean_source_macro_f1"],
    source_f1[expected_best_index],
    rel_tol=0.0,
    abs_tol=1e-12,
)
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip_norms = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
clipped_fractions = [
    float(row["gradient_clipped_fraction"])
    for row in history
]

assert max(post_clip_norms) <= 20.0002
assert all(0.0 <= value <= 1.0 for value in clipped_fractions)

best_checkpoint = torch.load(
    RUN_DIR / "best.pt",
    map_location="cpu",
    weights_only=False,
)
last_checkpoint = torch.load(
    RUN_DIR / "last.pt",
    map_location="cpu",
    weights_only=False,
)
initialization = torch.load(
    INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)

assert best_checkpoint["target_labels_used"] is False
assert last_checkpoint["target_labels_used"] is False

bn_suffixes = (
    "running_mean",
    "running_var",
    "num_batches_tracked",
)

for name, initial_value in initialization["state_dict"].items():
    if name.endswith(bn_suffixes):
        assert torch.equal(
            initial_value,
            best_checkpoint["model_state"][name],
        ), f"BatchNorm buffer changed: {name}"

print()
print("TRAINING HISTORY — SOURCE INFORMATION ONLY")
print(
    "epoch | cls_loss | mmd_loss | median_d2 | "
    "grad_before | grad_after | clipped_frac | mean_source_F1"
)

for row in history:
    print(
        f"{int(row['epoch']):5d} | "
        f"{float(row['classification_loss']):8.4f} | "
        f"{float(row['mmd_loss']):8.4f} | "
        f"{float(row['mmd_median_squared_distance']):9.6f} | "
        f"{float(row['gradient_norm']):11.4f} | "
        f"{float(row['gradient_norm_after_clipping']):10.4f} | "
        f"{float(row['gradient_clipped_fraction']):12.4f} | "
        f"{float(row['mean_source_macro_f1']):14.4f}"
    )

# Recompute the frozen selected checkpoint's source predictions.
protocol = load_protocol(PROTOCOL)
device = torch.device("cuda")

model = PACSClassifier(pretrained=False)
model.load_state_dict(best_checkpoint["model_state"])
model.to(device)
model.eval()

pooled_counts = Counter()

print()
print("SELECTED CHECKPOINT SOURCE PREDICTIONS")
print("Selected epoch:", best_checkpoint["epoch"])

for domain in SOURCES:
    loader = make_validation_loader(
        PACS_ROOT,
        protocol["source_splits"][domain]["validation"],
        batch_size=64,
        num_workers=2,
        pin_memory=True,
    )

    truth = []
    predictions = []

    with torch.inference_mode():
        for images, labels, _ in loader:
            logits, _ = model(images.to(device, non_blocking=True))
            predicted = logits.argmax(dim=1).cpu().tolist()

            truth.extend(int(value) for value in labels.tolist())
            predictions.extend(int(value) for value in predicted)

    counts = Counter(predictions)
    pooled_counts.update(predictions)

    accuracy = accuracy_score(truth, predictions)
    macro_f1 = f1_score(
        truth,
        predictions,
        labels=list(range(7)),
        average="macro",
        zero_division=0,
    )

    assert abs(
        accuracy - best_record[f"{domain}_accuracy"]
    ) < 1e-12
    assert abs(
        macro_f1 - best_record[f"{domain}_macro_f1"]
    ) < 1e-12

    print(
        domain,
        "counts=", dict(sorted(counts.items())),
        "accuracy=", round(accuracy, 6),
        "macro-F1=", round(macro_f1, 6),
    )

print()
print("POOLED COUNTS:", dict(sorted(pooled_counts.items())))
print()
print("NORMALIZED-MMD V3 DAN lambda_MMD=1 VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Maximum post-clipping gradient norm:",
    max(post_clip_norms),
)
print(
    "Mean clipped-step fraction:",
    sum(clipped_fractions) / len(clipped_fractions),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("BatchNorm buffers unchanged: True")
print("Checkpoint selection reconstructed: True")
print("Target labels accessed: False")
'''

subprocess.run(
    [sys.executable, "-c", verification_program],
    cwd=CODE_ROOT,
    check=True,
)

RuntimeError: /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/dan_1 already exists. Do not delete or overwrite it.

In [ ]:
from pathlib import Path
import json

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

paths = {
    "v3_code": CODE_ROOT,
    "v3_manifest": CODE_ROOT / "task2/PACKAGE-MANIFEST.json",
    "preflight": OUTPUT_ROOT / "preflight.json",
    "experiment_lock": OUTPUT_ROOT / "experiment_lock.json",
    "dan_1_directory": OUTPUT_ROOT / "dan_1",
    "dan_1_run_record": OUTPUT_ROOT / "dan_1/run.json",
}

for name, path in paths.items():
    print(f"{name}: {path.exists()} — {path}")

if paths["v3_manifest"].is_file():
    manifest = json.loads(paths["v3_manifest"].read_text())
    print("Manifest version:", manifest["version"])

if paths["preflight"].is_file():
    preflight = json.loads(paths["preflight"].read_text())
    print("Preflight status:", preflight["status"])
    print("Preflight target labels accessed:", preflight["target_labels_accessed"])

if paths["dan_1_run_record"].is_file():
    run = json.loads(paths["dan_1_run_record"].read_text())
    print("DAN-1 epochs:", run["epochs_completed"])
    print("DAN-1 best epoch:", run["best_epoch"])
    print("DAN-1 source macro-F1:", run["best_mean_source_macro_f1"])
    print("DAN-1 target labels used:", run["target_labels_used"])

v3_code: True — /content/task2-corrected-normalized-v3
v3_manifest: True — /content/task2-corrected-normalized-v3/task2/PACKAGE-MANIFEST.json
preflight: True — /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/preflight.json
experiment_lock: True — /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/experiment_lock.json
dan_1_directory: True — /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/dan_1
dan_1_run_record: False — /content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/dan_1/run.json
Manifest version: 3
Preflight status: PREFLIGHT_PASS
Preflight target labels accessed: False


In [ ]:
from pathlib import Path
import csv

RUN_DIR = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923/dan_1"
)

for name in [
    "best.pt",
    "last.pt",
    "history.csv",
    "best_source_validation.json",
    "run.json",
]:
    path = RUN_DIR / name
    print(
        f"{name}: exists={path.is_file()}",
        f"size={path.stat().st_size if path.is_file() else None}",
    )

history_path = RUN_DIR / "history.csv"

if history_path.is_file():
    with history_path.open(newline="") as handle:
        history = list(csv.DictReader(handle))

    print("Completed epochs:", len(history))
    if history:
        print("Last completed epoch:", history[-1]["epoch"])
        print(
            "Last source macro-F1:",
            history[-1]["mean_source_macro_f1"],
        )
else:
    print("Completed epochs: 0")

print("Target labels accessed: False")

best.pt: exists=True size=44803851
last.pt: exists=True size=134311699
history.csv: exists=True size=925
best_source_validation.json: exists=True size=367
run.json: exists=False size=None
Completed epochs: 2
Last completed epoch: 2
Last source macro-F1: 0.8312012565856631
Target labels accessed: False


In [ ]:
# RESUME V3 DAN lambda_MMD=1 FROM THE LAST COMPLETE EPOCH

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
RUN_DIR = OUTPUT_ROOT / "dan_1"

PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

assert (RUN_DIR / "last.pt").is_file()
assert not (RUN_DIR / "run.json").exists()

print("Resuming normalized-MMD v3 DAN lambda_MMD=1.")
print("Resume point: epoch 2")
print("Target labels accessed: False")

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id",
        "dan_1",
        "--pacs-root",
        str(PACS_ROOT),
        "--dataset-source",
        (
            "Google Drive PACS_dassl.zip; "
            "sha256="
            "0dc9d0176fa27c9b4504e7c2e962aebe"
            "6a79ed0c1819b84148786e590f87e102"
        ),
        "--protocol",
        str(PROTOCOL),
        "--initialization",
        str(INITIALIZATION),
        "--preregistration",
        str(PREREGISTRATION),
        "--output",
        str(OUTPUT_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
        "--resume",
    ],
    cwd=CODE_ROOT,
    check=True,
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

assert run_record["target_labels_used"] is False
assert run_record["identity"]["config"]["mmd_lambda"] == 1.0
assert (
    run_record["identity"]["config"]["mmd_feature_normalization"]
    == "l2_per_sample"
)
assert run_record["identity"]["config"]["gradient_clipping"] == 20.0
assert len(history) == run_record["epochs_completed"]

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value))

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
assert max(post_clip) <= 20.0002

print()
print("V3 DAN lambda_MMD=1 TRAINING HISTORY")
print(
    "epoch | cls_loss | mmd_loss | median_d2 | "
    "grad_before | clipped_frac | source_F1"
)

for row in history:
    print(
        f"{int(row['epoch']):5d} | "
        f"{float(row['classification_loss']):8.4f} | "
        f"{float(row['mmd_loss']):8.4f} | "
        f"{float(row['mmd_median_squared_distance']):9.6f} | "
        f"{float(row['gradient_norm']):11.4f} | "
        f"{float(row['gradient_clipped_fraction']):12.4f} | "
        f"{float(row['mean_source_macro_f1']):9.4f}"
    )

print()
print("V3 DAN lambda_MMD=1 VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Mean clipped-step fraction:",
    sum(
        float(row["gradient_clipped_fraction"])
        for row in history
    ) / len(history),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("Target labels accessed: False")

Resuming normalized-MMD v3 DAN lambda_MMD=1.
Resume point: epoch 2
Target labels accessed: False

V3 DAN lambda_MMD=1 TRAINING HISTORY
epoch | cls_loss | mmd_loss | median_d2 | grad_before | clipped_frac | source_F1
    1 |   0.4922 |   0.1606 |  0.198120 |     16.8605 |       0.2638 |    0.8851
    2 |   0.2352 |   0.1333 |  0.030030 |     23.3881 |       0.5574 |    0.8312
    3 |   0.1695 |   0.1238 |  0.007824 |     34.4920 |       0.7106 |    0.8986
    4 |   0.1488 |   0.1241 |  0.002325 |     61.9858 |       0.8000 |    0.8176
    5 |   0.1471 |   0.1264 |  0.001453 |     75.3759 |       0.8340 |    0.8843
    6 |   0.1801 |   0.1217 |  0.000754 |    127.1830 |       0.8468 |    0.9061
    7 |   0.1593 |   0.1284 |  0.000668 |    130.4750 |       0.8638 |    0.7493
    8 |   0.1534 |   0.1256 |  0.000342 |    162.2379 |       0.8766 |    0.7347
    9 |   0.1821 |   0.1269 |  0.000228 |    242.6160 |       0.8979 |    0.7858
   10 |   0.1591 |   0.1252 |  0.000199 |    223.2150 |

In [30]:
# TRAIN OR RESUME OFFICIAL V3 SOURCE-ONLY

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

RUN_ID = "source_only"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

command = [
    sys.executable,
    "-m",
    "task2.train",
    "--run-id",
    RUN_ID,
    "--pacs-root",
    str(PACS_ROOT),
    "--dataset-source",
    (
        "Google Drive PACS_dassl.zip; "
        "sha256="
        "0dc9d0176fa27c9b4504e7c2e962aebe"
        "6a79ed0c1819b84148786e590f87e102"
    ),
    "--protocol",
    str(PROTOCOL),
    "--initialization",
    str(INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--output",
    str(OUTPUT_ROOT),
    "--device",
    "cuda",
    "--num-workers",
    "2",
]

if (RUN_DIR / "run.json").is_file():
    print("Source-only is already complete; verifying it.")
elif RUN_DIR.exists():
    if not (RUN_DIR / "last.pt").is_file():
        raise RuntimeError(
            "Source-only directory exists without a resumable checkpoint."
        )
    print("Resuming Source-only from its last complete epoch.")
    subprocess.run(
        command + ["--resume"],
        cwd=CODE_ROOT,
        check=True,
    )
else:
    print("Starting official v3 Source-only.")
    subprocess.run(command, cwd=CODE_ROOT, check=True)

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

assert run_record["run_id"] == RUN_ID
assert run_record["target_labels_used"] is False
assert run_record["identity"]["config"]["method"] == "source_only"
assert run_record["identity"]["config"]["gradient_clipping"] == 20.0
assert len(history) == run_record["epochs_completed"]

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value))

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
assert max(post_clip) <= 20.0002

print()
print("OFFICIAL V3 SOURCE-ONLY VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Mean clipped-step fraction:",
    sum(
        float(row["gradient_clipped_fraction"])
        for row in history
    ) / len(history),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("Target images used for training: False")
print("Target labels accessed: False")

Source-only is already complete; verifying it.

OFFICIAL V3 SOURCE-ONLY VERIFIED COMPLETE
Epochs completed: 9
Selected epoch: 4
Best mean source-validation macro-F1: 0.9426262342459099
Mean clipped-step fraction: 0.017966903073286054
Checkpoint SHA256: 3d28a223e4b97b323cb3a20dcb5b7577af96631f2e6ef1f2bc99d53d85761327
Target images used for training: False
Target labels accessed: False


In [31]:
# TRAIN OR RESUME OFFICIAL V3 DAN lambda_MMD=0.1

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

RUN_ID = "dan_0p1"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

command = [
    sys.executable,
    "-m",
    "task2.train",
    "--run-id",
    RUN_ID,
    "--pacs-root",
    str(PACS_ROOT),
    "--dataset-source",
    (
        "Google Drive PACS_dassl.zip; "
        "sha256="
        "0dc9d0176fa27c9b4504e7c2e962aebe"
        "6a79ed0c1819b84148786e590f87e102"
    ),
    "--protocol",
    str(PROTOCOL),
    "--initialization",
    str(INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--output",
    str(OUTPUT_ROOT),
    "--device",
    "cuda",
    "--num-workers",
    "2",
]

if (RUN_DIR / "run.json").is_file():
    print("DAN lambda_MMD=0.1 is already complete; verifying it.")
elif RUN_DIR.exists():
    if not (RUN_DIR / "last.pt").is_file():
        raise RuntimeError(
            "DAN 0.1 directory exists without a resumable checkpoint."
        )
    print("Resuming DAN lambda_MMD=0.1.")
    subprocess.run(command + ["--resume"], cwd=CODE_ROOT, check=True)
else:
    print("Starting official v3 DAN lambda_MMD=0.1.")
    subprocess.run(command, cwd=CODE_ROOT, check=True)

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

config = run_record["identity"]["config"]

assert run_record["run_id"] == RUN_ID
assert run_record["target_labels_used"] is False
assert config["method"] == "dan"
assert config["mmd_lambda"] == 0.1
assert config["mmd_feature_normalization"] == "l2_per_sample"
assert config["gradient_clipping"] == 20.0
assert len(history) == run_record["epochs_completed"]

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value))

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
assert max(post_clip) <= 20.0002

print()
print("OFFICIAL V3 DAN lambda_MMD=0.1 VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Minimum epoch-average MMD median:",
    min(
        float(row["mmd_median_squared_distance"])
        for row in history
    ),
)
print(
    "Mean clipped-step fraction:",
    sum(
        float(row["gradient_clipped_fraction"])
        for row in history
    ) / len(history),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("Target labels accessed: False")

Starting official v3 DAN lambda_MMD=0.1.

OFFICIAL V3 DAN lambda_MMD=0.1 VERIFIED COMPLETE
Epochs completed: 7
Selected epoch: 2
Best mean source-validation macro-F1: 0.9320428032141853
Minimum epoch-average MMD median: 0.521780781796638
Mean clipped-step fraction: 0.02006079027355623
Checkpoint SHA256: 4d0d7132d25404bfab98446ad49e116695d90cc1f2d6a7750514de4da14d4499
Target labels accessed: False


In [32]:
# TRAIN OR RESUME OFFICIAL V3 DAN lambda_MMD=10

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

RUN_ID = "dan_10"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

command = [
    sys.executable,
    "-m",
    "task2.train",
    "--run-id",
    RUN_ID,
    "--pacs-root",
    str(PACS_ROOT),
    "--dataset-source",
    (
        "Google Drive PACS_dassl.zip; "
        "sha256="
        "0dc9d0176fa27c9b4504e7c2e962aebe"
        "6a79ed0c1819b84148786e590f87e102"
    ),
    "--protocol",
    str(PROTOCOL),
    "--initialization",
    str(INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--output",
    str(OUTPUT_ROOT),
    "--device",
    "cuda",
    "--num-workers",
    "2",
]

if (RUN_DIR / "run.json").is_file():
    print("DAN lambda_MMD=10 is already complete; verifying it.")
elif RUN_DIR.exists():
    if not (RUN_DIR / "last.pt").is_file():
        raise RuntimeError(
            "DAN 10 directory exists without a resumable checkpoint."
        )
    print("Resuming DAN lambda_MMD=10.")
    subprocess.run(command + ["--resume"], cwd=CODE_ROOT, check=True)
else:
    print("Starting official v3 DAN lambda_MMD=10.")
    subprocess.run(command, cwd=CODE_ROOT, check=True)

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

config = run_record["identity"]["config"]

assert run_record["run_id"] == RUN_ID
assert run_record["target_labels_used"] is False
assert config["method"] == "dan"
assert config["mmd_lambda"] == 10.0
assert config["mmd_feature_normalization"] == "l2_per_sample"
assert config["gradient_clipping"] == 20.0
assert len(history) == run_record["epochs_completed"]

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value))

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
assert max(post_clip) <= 20.0002

print()
print("OFFICIAL V3 DAN lambda_MMD=10 VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Minimum epoch-average MMD median:",
    min(
        float(row["mmd_median_squared_distance"])
        for row in history
    ),
)
print(
    "Maximum pre-clipping gradient norm:",
    max(float(row["gradient_norm"]) for row in history),
)
print(
    "Mean clipped-step fraction:",
    sum(
        float(row["gradient_clipped_fraction"])
        for row in history
    ) / len(history),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("Target labels accessed: False")

Starting official v3 DAN lambda_MMD=10.

OFFICIAL V3 DAN lambda_MMD=10 VERIFIED COMPLETE
Epochs completed: 6
Selected epoch: 1
Best mean source-validation macro-F1: 0.05066996495567924
Minimum epoch-average MMD median: 1.071434732543351e-09
Maximum pre-clipping gradient norm: 1620.4299275763478
Mean clipped-step fraction: 1.0
Checkpoint SHA256: 487dc643227bae3f9057ccb7f05a282a252cadbd9471d38dbb78fbb6d30ea5aa
Target labels accessed: False


In [34]:
# TRAIN OR RESUME OFFICIAL V3 DANN

from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)

RUN_ID = "dann"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PROTOCOL = CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
PREREGISTRATION = (
    CODE_ROOT / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)
INITIALIZATION = (
    OUTPUT_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

command = [
    sys.executable,
    "-m",
    "task2.train",
    "--run-id",
    RUN_ID,
    "--pacs-root",
    str(PACS_ROOT),
    "--dataset-source",
    (
        "Google Drive PACS_dassl.zip; "
        "sha256="
        "0dc9d0176fa27c9b4504e7c2e962aebe"
        "6a79ed0c1819b84148786e590f87e102"
    ),
    "--protocol",
    str(PROTOCOL),
    "--initialization",
    str(INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--output",
    str(OUTPUT_ROOT),
    "--device",
    "cuda",
    "--num-workers",
    "2",
]

if (RUN_DIR / "run.json").is_file():
    print("DANN is already complete; verifying it.")
elif RUN_DIR.exists():
    if not (RUN_DIR / "last.pt").is_file():
        raise RuntimeError(
            "DANN directory exists without a resumable checkpoint."
        )
    print("Resuming official v3 DANN.")
    subprocess.run(command + ["--resume"], cwd=CODE_ROOT, check=True)
else:
    print("Starting official v3 DANN.")
    subprocess.run(command, cwd=CODE_ROOT, check=True)

run_record = json.loads((RUN_DIR / "run.json").read_text())
best_record = json.loads(
    (RUN_DIR / "best_source_validation.json").read_text()
)

with (RUN_DIR / "history.csv").open(newline="") as handle:
    history = list(csv.DictReader(handle))

config = run_record["identity"]["config"]

assert run_record["run_id"] == RUN_ID
assert run_record["target_labels_used"] is False
assert config["method"] == "dann"
assert config["domain_loss_weight"] == 1.0
assert config["gradient_clipping"] == 20.0
assert len(history) == run_record["epochs_completed"]

for row in history:
    for name, value in row.items():
        if name != "epoch":
            assert math.isfinite(float(value))

source_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
best_index = max(
    range(len(source_f1)),
    key=lambda index: source_f1[index],
)
expected_best_epoch = int(history[best_index]["epoch"])

assert run_record["best_epoch"] == expected_best_epoch
assert best_record["epoch"] == expected_best_epoch
assert run_record["best_checkpoint_sha256"] == sha256_file(
    RUN_DIR / "best.pt"
)

post_clip = [
    float(row["gradient_norm_after_clipping"])
    for row in history
]
assert max(post_clip) <= 20.0002

print()
print("OFFICIAL V3 DANN VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", run_record["best_epoch"])
print(
    "Best mean source-validation macro-F1:",
    run_record["best_mean_source_macro_f1"],
)
print(
    "Maximum pre-clipping gradient norm:",
    max(float(row["gradient_norm"]) for row in history),
)
print(
    "Mean clipped-step fraction:",
    sum(
        float(row["gradient_clipped_fraction"])
        for row in history
    ) / len(history),
)
print(
    "Selected-epoch training domain accuracy:",
    float(history[expected_best_index]["domain_accuracy"]),
)
print(
    "Selected-epoch GRL strength:",
    float(history[expected_best_index]["grl_strength"]),
)
print("Checkpoint SHA256:", run_record["best_checkpoint_sha256"])
print("Target labels accessed: False")

Starting official v3 DANN.

OFFICIAL V3 DANN VERIFIED COMPLETE
Epochs completed: 6
Selected epoch: 1
Best mean source-validation macro-F1: 0.6049088767773435
Maximum pre-clipping gradient norm: 40455498.998820186
Mean clipped-step fraction: 0.9340425531914893
Selected-epoch training domain accuracy: 0.5203014316235451
Selected-epoch GRL strength: 0.08261060619425821
Checkpoint SHA256: 321e3389a246ad179d8219584d04f9b089c8a13d7ef5512d34bceab4f0f676e4
Target labels accessed: False


In [35]:
# V4 ADVERSARIAL-NORMALIZATION PILOT — INSTALL AND PREFLIGHT ONLY

from google.colab import drive
drive.mount("/content/drive")

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import sklearn
import torch
import torchvision

REPOSITORY = "https://github.com/therealshaheer11-glithc/ATML-Assignment-1.git"

CLONE_ROOT = Path("/content/task2-v4-github")
CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

V3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

V3_INITIALIZATION = (
    V3_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
V4_INITIALIZATION = (
    V4_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

EXPECTED_MANIFEST_SHA256 = (
    "3cec075fea3295ece5ff42ca518424723497b95995bd58e8840a0bc13584c01c"
)
EXPECTED_INITIALIZATION_STATE_SHA256 = (
    "567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622"
)

# Require the same environment used for v3.
actual_environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}

expected_environment = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "numpy": "2.1.3",
    "sklearn": "1.6.1",
    "cuda": "12.8",
    "cudnn": 91900,
    "gpu": "Tesla T4",
}

if actual_environment != expected_environment:
    raise RuntimeError(
        "V4 PREFLIGHT STOP: environment differs from v3.\n"
        + json.dumps(
            {
                "expected": expected_environment,
                "actual": actual_environment,
            },
            indent=2,
        )
    )

if not PACS_ROOT.is_dir():
    raise FileNotFoundError(
        f"PACS data is missing from {PACS_ROOT}. "
        "Rerun the existing PACS extraction cell first."
    )

if not V3_INITIALIZATION.is_file():
    raise FileNotFoundError(
        "The locked v3 common initialization is missing:\n"
        f"{V3_INITIALIZATION}"
    )

# Refresh only temporary local copies. Drive results remain untouched.
for temporary_path in (CLONE_ROOT, CODE_ROOT):
    if temporary_path.exists():
        shutil.rmtree(temporary_path)

subprocess.run(
    ["git", "clone", "--depth", "1", REPOSITORY, str(CLONE_ROOT)],
    check=True,
)

github_commit = subprocess.check_output(
    ["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

for folder_name in ("shared", "task2", "tests"):
    source = CLONE_ROOT / folder_name
    if not source.is_dir():
        raise FileNotFoundError(
            f"GitHub checkout is missing required folder: {folder_name}"
        )
    shutil.copytree(source, CODE_ROOT / folder_name)

manifest_path = CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
manifest_bytes = manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()

if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        "V4 manifest differs from the approved package.\n"
        f"Expected: {EXPECTED_MANIFEST_SHA256}\n"
        f"Actual:   {manifest_sha256}"
    )

manifest = json.loads(manifest_bytes)

assert manifest["version"] == "4"
assert manifest["training_started"] is False
assert manifest["target_evaluation_included"] is False
assert manifest["status"] == (
    "candidate_pilot_pending_source_only_adoption_decision"
)

# Verify every governed file against the committed manifest.
expected_files = {
    item["path"]: item
    for item in manifest["files"]
}

actual_files = {}
for base_name in ("shared", "task2", "tests"):
    for path in (CODE_ROOT / base_name).rglob("*"):
        if (
            path.is_file()
            and path != manifest_path
            and path.suffix in {".py", ".json", ".md", ".txt"}
        ):
            actual_files[path.relative_to(CODE_ROOT).as_posix()] = path

if set(actual_files) != set(expected_files):
    raise RuntimeError(
        "The committed governed-file set does not match the v4 manifest."
    )

for relative_path, record in expected_files.items():
    data = actual_files[relative_path].read_bytes()
    if len(data) != record["bytes"]:
        raise RuntimeError(f"Size mismatch: {relative_path}")
    if hashlib.sha256(data).hexdigest() != record["sha256"]:
        raise RuntimeError(f"SHA256 mismatch: {relative_path}")

# Compile and run the nine locked protocol tests.
subprocess.run(
    [sys.executable, "-m", "compileall", "-q", str(CODE_ROOT)],
    check=True,
)

sys.path.insert(0, str(CODE_ROOT))

test_path = CODE_ROOT / "tests/test_locked_choices.py"
spec = importlib.util.spec_from_file_location(
    "v4_locked_tests",
    test_path,
)
test_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_module)

tests_run = 0
for name in sorted(dir(test_module)):
    if name.startswith("test_"):
        getattr(test_module, name)()
        tests_run += 1

if tests_run != 9:
    raise RuntimeError(f"Expected 9 tests; executed {tests_run}")

from task2.config import RUNS, load_config
from task2.model import state_dict_sha256

for run_id in RUNS:
    config = load_config(run_id)
    assert config["adversarial_feature_normalization"] == "l2_per_sample"
    assert config["gradient_clipping"] == 20.0

# Copy and independently verify the exact v3 initialization.
V4_INITIALIZATION.parent.mkdir(parents=True, exist_ok=True)

if not V4_INITIALIZATION.exists():
    shutil.copy2(V3_INITIALIZATION, V4_INITIALIZATION)

initialization_payload = torch.load(
    V4_INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)

initialization_state_sha256 = state_dict_sha256(
    initialization_payload["state_dict"]
)

if (
    initialization_state_sha256
    != EXPECTED_INITIALIZATION_STATE_SHA256
):
    raise RuntimeError(
        "Common initialization state differs from v3."
    )

# Record the exact GitHub commit used by this pilot.
V4_ROOT.mkdir(parents=True, exist_ok=True)

(V4_ROOT / "github_commit.json").write_text(
    json.dumps(
        {
            "repository": REPOSITORY,
            "commit": github_commit,
            "manifest_sha256": manifest_sha256,
            "training_started": False,
            "target_labels_accessed": False,
        },
        indent=2,
    )
    + "\n"
)

# Dataset/configuration preflight. This performs no training.
preflight_path = V4_ROOT / "preflight.json"

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared/splits/pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_path),
    ],
    cwd=CODE_ROOT,
    check=True,
)

preflight = json.loads(preflight_path.read_text())

assert preflight["status"] == "PREFLIGHT_PASS"
assert preflight["training_started"] is False
assert preflight["target_labels_accessed"] is False

print("\nV4 PILOT PREFLIGHT COMPLETE")
print("GitHub commit:", github_commit)
print("Manifest SHA256:", manifest_sha256)
print("Tests passed:", tests_run)
print("Initialization SHA256:", initialization_state_sha256)
print("GPU:", actual_environment["gpu"])
print("Training started:", False)
print("Target labels accessed:", False)
print("Code root:", CODE_ROOT)
print("Pilot output root:", V4_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ImportError: cannot import name 'l2_normalize_alignment_features' from 'shared.mmd' (/content/task2-corrected-code/shared/mmd.py)

In [36]:
# V4 PREFLIGHT RECOVERY — clear cached v3 imports and finish setup

import hashlib
import importlib
import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

CLONE_ROOT = Path("/content/task2-v4-github")
CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

V3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

V3_INITIALIZATION = (
    V3_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
V4_INITIALIZATION = (
    V4_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

EXPECTED_MANIFEST_SHA256 = (
    "3cec075fea3295ece5ff42ca518424723497b95995bd58e8840a0bc13584c01c"
)
EXPECTED_INITIALIZATION_STATE_SHA256 = (
    "567ae5fee162cb523261192c469b24e0c65fd793cb0484e0c0fc14367d85f622"
)

if not CODE_ROOT.is_dir():
    raise FileNotFoundError(
        "The previous setup did not create the v4 code directory."
    )

if not PACS_ROOT.is_dir():
    raise FileNotFoundError(f"PACS data missing: {PACS_ROOT}")

if not V3_INITIALIZATION.is_file():
    raise FileNotFoundError(
        f"V3 initialization missing: {V3_INITIALIZATION}"
    )

# Remove cached imports from the earlier code versions.
for module_name in list(sys.modules):
    if (
        module_name == "shared"
        or module_name.startswith("shared.")
        or module_name == "task2"
        or module_name.startswith("task2.")
        or module_name == "tests"
        or module_name.startswith("tests.")
        or module_name == "v4_locked_tests"
    ):
        del sys.modules[module_name]

# Remove previous Task 2 code paths and prioritize v4.
sys.path = [
    entry
    for entry in sys.path
    if not (
        isinstance(entry, str)
        and entry.startswith("/content/task2")
    )
]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()

# Confirm Python is now loading v4.
import shared.mmd

loaded_mmd_path = Path(shared.mmd.__file__).resolve()
expected_mmd_path = (CODE_ROOT / "shared/mmd.py").resolve()

if loaded_mmd_path != expected_mmd_path:
    raise RuntimeError(
        "Python still loaded the wrong MMD module:\n"
        f"{loaded_mmd_path}"
    )

assert hasattr(
    shared.mmd,
    "l2_normalize_alignment_features",
)

# Reverify the committed manifest.
manifest_path = CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        "V4 manifest mismatch.\n"
        f"Expected: {EXPECTED_MANIFEST_SHA256}\n"
        f"Actual:   {manifest_sha256}"
    )

manifest = json.loads(manifest_path.read_text())
assert manifest["version"] == "4"
assert manifest["training_started"] is False
assert manifest["target_evaluation_included"] is False

# Run all nine v4 tests.
test_path = CODE_ROOT / "tests/test_locked_choices.py"
spec = importlib.util.spec_from_file_location(
    "v4_locked_tests",
    test_path,
)
test_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_module)

tests_run = 0
for name in sorted(dir(test_module)):
    if name.startswith("test_"):
        getattr(test_module, name)()
        tests_run += 1

if tests_run != 9:
    raise RuntimeError(f"Expected 9 tests; ran {tests_run}")

from task2.config import RUNS, load_config
from task2.model import state_dict_sha256

for run_id in RUNS:
    config = load_config(run_id)
    assert config["adversarial_feature_normalization"] == "l2_per_sample"
    assert config["gradient_clipping"] == 20.0

# Reuse and verify the exact v3 common initialization.
V4_INITIALIZATION.parent.mkdir(parents=True, exist_ok=True)

if not V4_INITIALIZATION.exists():
    shutil.copy2(V3_INITIALIZATION, V4_INITIALIZATION)

initialization_payload = torch.load(
    V4_INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)
initialization_state_sha256 = state_dict_sha256(
    initialization_payload["state_dict"]
)

if initialization_state_sha256 != EXPECTED_INITIALIZATION_STATE_SHA256:
    raise RuntimeError("V4 initialization does not match v3.")

github_commit = subprocess.check_output(
    ["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

V4_ROOT.mkdir(parents=True, exist_ok=True)
(V4_ROOT / "github_commit.json").write_text(
    json.dumps(
        {
            "repository": (
                "https://github.com/"
                "therealshaheer11-glithc/ATML-Assignment-1"
            ),
            "commit": github_commit,
            "manifest_sha256": manifest_sha256,
            "training_started": False,
            "target_labels_accessed": False,
        },
        indent=2,
    )
    + "\n"
)

# Complete the read-only preflight.
preflight_path = V4_ROOT / "preflight.json"

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared/splits/pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_path),
    ],
    cwd=CODE_ROOT,
    check=True,
)

preflight = json.loads(preflight_path.read_text())
assert preflight["status"] == "PREFLIGHT_PASS"
assert preflight["training_started"] is False
assert preflight["target_labels_accessed"] is False

print("\nV4 PILOT PREFLIGHT COMPLETE")
print("Loaded MMD module:", loaded_mmd_path)
print("GitHub commit:", github_commit)
print("Manifest SHA256:", manifest_sha256)
print("Tests passed:", tests_run)
print("Initialization SHA256:", initialization_state_sha256)
print("Preflight:", preflight["status"])
print("Training started:", False)
print("Target labels accessed:", False)
print("Pilot output root:", V4_ROOT)

RuntimeError: V4 initialization does not match v3.

In [37]:
# VERIFY V3 INITIALIZATION FROM ALL COMPLETED RUNS, THEN FINISH V4 PREFLIGHT

import hashlib
import importlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

CLONE_ROOT = Path("/content/task2-v4-github")
CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

V3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

V3_INITIALIZATION = (
    V3_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
V4_INITIALIZATION = (
    V4_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)

COMPLETED_V3_RUNS = [
    "source_only",
    "dan_0p1",
    "dan_1",
    "dan_10",
    "dann",
]

# Clear older cached code imports again.
for module_name in list(sys.modules):
    if (
        module_name == "shared"
        or module_name.startswith("shared.")
        or module_name == "task2"
        or module_name.startswith("task2.")
    ):
        del sys.modules[module_name]

sys.path = [
    entry
    for entry in sys.path
    if not (
        isinstance(entry, str)
        and entry.startswith("/content/task2")
    )
]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()

from task2.model import state_dict_sha256

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not V3_INITIALIZATION.is_file():
    raise FileNotFoundError(V3_INITIALIZATION)

# Verify the initialization file internally.
v3_payload = torch.load(
    V3_INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)
calculated_state_sha256 = state_dict_sha256(
    v3_payload["state_dict"]
)
declared_state_sha256 = v3_payload.get("state_dict_sha256")

if declared_state_sha256 != calculated_state_sha256:
    raise RuntimeError(
        "The v3 initialization file fails its own internal hash check."
    )

# Independently confirm every completed v3 run recorded this same initialization.
run_hashes = {}

for run_id in COMPLETED_V3_RUNS:
    run_record_path = V3_ROOT / run_id / "run.json"

    if not run_record_path.is_file():
        raise FileNotFoundError(
            f"Missing completed v3 run record: {run_record_path}"
        )

    run_record = json.loads(run_record_path.read_text())
    recorded_hash = run_record["identity"]["initialization_sha256"]
    run_hashes[run_id] = recorded_hash

unique_run_hashes = set(run_hashes.values())

if unique_run_hashes != {calculated_state_sha256}:
    raise RuntimeError(
        "The completed v3 runs do not unanimously match the "
        "stored v3 initialization:\n"
        + json.dumps(run_hashes, indent=2)
    )

# Copy the verified file without regenerating it.
V4_INITIALIZATION.parent.mkdir(parents=True, exist_ok=True)

if V4_INITIALIZATION.exists():
    if file_sha256(V4_INITIALIZATION) != file_sha256(V3_INITIALIZATION):
        raise RuntimeError(
            "Existing v4 initialization is not byte-identical to v3."
        )
else:
    shutil.copy2(V3_INITIALIZATION, V4_INITIALIZATION)

v4_payload = torch.load(
    V4_INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)
v4_state_sha256 = state_dict_sha256(v4_payload["state_dict"])

if v4_state_sha256 != calculated_state_sha256:
    raise RuntimeError("Copied v4 initialization failed verification.")

github_commit = subprocess.check_output(
    ["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

manifest_path = CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
manifest_sha256 = file_sha256(manifest_path)

V4_ROOT.mkdir(parents=True, exist_ok=True)

# Record exactly how the initialization was verified.
(V4_ROOT / "initialization_verification.json").write_text(
    json.dumps(
        {
            "v3_initialization_path": str(V3_INITIALIZATION),
            "v4_initialization_path": str(V4_INITIALIZATION),
            "initialization_state_sha256": calculated_state_sha256,
            "initialization_file_sha256": file_sha256(V3_INITIALIZATION),
            "completed_v3_run_references": run_hashes,
            "all_references_match": True,
            "target_labels_accessed": False,
        },
        indent=2,
    )
    + "\n"
)

(V4_ROOT / "github_commit.json").write_text(
    json.dumps(
        {
            "repository": (
                "https://github.com/"
                "therealshaheer11-glithc/ATML-Assignment-1"
            ),
            "commit": github_commit,
            "manifest_sha256": manifest_sha256,
            "training_started": False,
            "target_labels_accessed": False,
        },
        indent=2,
    )
    + "\n"
)

# Finish the read-only preflight.
preflight_path = V4_ROOT / "preflight.json"

subprocess.run(
    [
        sys.executable,
        "-m",
        "task2.preflight",
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            CODE_ROOT
            / "shared/splits/pacs_sketch_seed6304.json"
        ),
        "--output",
        str(preflight_path),
    ],
    cwd=CODE_ROOT,
    check=True,
)

preflight = json.loads(preflight_path.read_text())

assert preflight["status"] == "PREFLIGHT_PASS"
assert preflight["training_started"] is False
assert preflight["target_labels_accessed"] is False

print("\nV4 PILOT PREFLIGHT COMPLETE")
print("Authoritative initialization SHA256:", calculated_state_sha256)
print("All five completed v3 runs match:", True)
print("V3/V4 initialization files are byte-identical:", True)
print("GitHub commit:", github_commit)
print("Manifest SHA256:", manifest_sha256)
print("Preflight:", preflight["status"])
print("Training started:", False)
print("Target labels accessed:", False)
print("Pilot output root:", V4_ROOT)


V4 PILOT PREFLIGHT COMPLETE
Authoritative initialization SHA256: 4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3
All five completed v3 runs match: True
V3/V4 initialization files are byte-identical: True
GitHub commit: d26997b22d3b7722e2ecc828dde1445244afc04b
Manifest SHA256: 3cec075fea3295ece5ff42ca518424723497b95995bd58e8840a0bc13584c01c
Preflight: PREFLIGHT_PASS
Training started: False
Target labels accessed: False
Pilot output root: /content/drive/MyDrive/ATML-PA1/task2_adversarial_normalized_v4_20260923


In [38]:
# RUN V4 DANN PILOT — TARGET LABELS REMAIN INACCESSIBLE

import csv
import hashlib
import json
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

INITIALIZATION = (
    V4_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
PROTOCOL = (
    CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
)
PREREGISTRATION = (
    CODE_ROOT
    / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)

RUN_DIR = V4_ROOT / "dann"
RUN_RECORD = RUN_DIR / "run.json"
LAST_CHECKPOINT = RUN_DIR / "last.pt"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable.")

if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError(
        "This pilot requires the same Tesla T4 used for v3."
    )

for required_path in (
    CODE_ROOT,
    PACS_ROOT,
    INITIALIZATION,
    PROTOCOL,
    PREREGISTRATION,
    V4_ROOT / "preflight.json",
    V4_ROOT / "github_commit.json",
    V4_ROOT / "initialization_verification.json",
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

# If interruption occurred before epoch 1 completed, preserve that partial
# directory and begin again. Completed epochs are resumed normally.
if (
    RUN_DIR.exists()
    and not RUN_RECORD.exists()
    and not LAST_CHECKPOINT.exists()
):
    timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    preserved = V4_ROOT / f"dann_incomplete_before_epoch_{timestamp}"
    shutil.move(str(RUN_DIR), str(preserved))
    print("Preserved incomplete pre-epoch run at:", preserved)

if RUN_RECORD.exists():
    print("V4 DANN is already complete; verifying it.")
else:
    command = [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id",
        "dann",
        "--pacs-root",
        str(PACS_ROOT),
        "--dataset-source",
        (
            "Verified PACS archive; SHA256="
            "0dc9d0176fa27c9b4504e7c2e962aebe"
            "6a79ed0c1819b84148786e590f87e102"
        ),
        "--protocol",
        str(PROTOCOL),
        "--initialization",
        str(INITIALIZATION),
        "--preregistration",
        str(PREREGISTRATION),
        "--output",
        str(V4_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
    ]

    if LAST_CHECKPOINT.exists():
        command.append("--resume")
        print("Resuming v4 DANN from the last completed epoch.")
    else:
        print("Starting v4 DANN from the locked common initialization.")

    print("GPU:", torch.cuda.get_device_name(0))
    print("Target labels accessed:", False)

    subprocess.run(
        command,
        cwd=CODE_ROOT,
        check=True,
    )

if not RUN_RECORD.is_file():
    raise RuntimeError("Training ended without a completed run record.")

run_record = json.loads(RUN_RECORD.read_text())

if run_record["target_labels_used"] is not False:
    raise RuntimeError("Run record indicates target-label access.")

history_path = RUN_DIR / "history.csv"
with history_path.open(newline="") as handle:
    history = list(csv.DictReader(handle))

if len(history) != run_record["epochs_completed"]:
    raise RuntimeError("History length and run record disagree.")

def column(name):
    return [float(row[name]) for row in history]

post_normalization_norms = column(
    "adversarial_feature_norm_after"
)

if not all(
    np.isfinite(value) and abs(value - 1.0) < 1e-5
    for value in post_normalization_norms
):
    raise RuntimeError(
        "The discriminator-input normalization check failed."
    )

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

best_checkpoint = RUN_DIR / "best.pt"
actual_checkpoint_sha256 = file_sha256(best_checkpoint)

if (
    actual_checkpoint_sha256
    != run_record["best_checkpoint_sha256"]
):
    raise RuntimeError("Selected checkpoint hash mismatch.")

best_epoch = int(run_record["best_epoch"])
selected_row = next(
    row for row in history
    if int(row["epoch"]) == best_epoch
)

v3_source_f1 = 0.6049088767773435
v4_source_f1 = float(
    run_record["best_mean_source_macro_f1"]
)

print("\nV4 DANN SOURCE-ONLY TRAINING HISTORY")
print(
    "epoch | cls_loss | domain_loss | domain_acc | "
    "GRL | feature_norm_after | grad_before | clipped_frac | source_F1"
)

for row in history:
    print(
        f"{int(row['epoch']):5d} | "
        f"{float(row['classification_loss']):8.4f} | "
        f"{float(row['domain_loss']):11.4f} | "
        f"{float(row['domain_accuracy']):10.4f} | "
        f"{float(row['grl_strength']):5.4f} | "
        f"{float(row['adversarial_feature_norm_after']):18.6f} | "
        f"{float(row['gradient_norm']):11.4f} | "
        f"{float(row['gradient_clipped_fraction']):12.4f} | "
        f"{float(row['mean_source_macro_f1']):9.4f}"
    )

print("\nV4 DANN PILOT VERIFIED COMPLETE")
print("Epochs completed:", run_record["epochs_completed"])
print("Selected epoch:", best_epoch)
print("Best v4 source-validation macro-F1:", v4_source_f1)
print("Previous v3 DANN source macro-F1:", v3_source_f1)
print("Source-F1 change:", v4_source_f1 - v3_source_f1)
print(
    "Maximum pre-clipping gradient norm:",
    max(column("gradient_norm")),
)
print(
    "Mean clipped-step fraction:",
    float(np.mean(column("gradient_clipped_fraction"))),
)
print(
    "Selected-epoch domain accuracy:",
    float(selected_row["domain_accuracy"]),
)
print(
    "Selected-epoch GRL strength:",
    float(selected_row["grl_strength"]),
)
print(
    "Mean discriminator-input norm after normalization:",
    float(np.mean(post_normalization_norms)),
)
print("Checkpoint SHA256:", actual_checkpoint_sha256)
print("Target labels accessed:", False)
print("Adoption decision made automatically:", False)

Starting v4 DANN from the locked common initialization.
GPU: Tesla T4
Target labels accessed: False

V4 DANN SOURCE-ONLY TRAINING HISTORY
epoch | cls_loss | domain_loss | domain_acc | GRL | feature_norm_after | grad_before | clipped_frac | source_F1
    1 |   0.4660 |      0.6369 |     0.7687 | 0.0826 |           1.000000 |     11.7314 |       0.0468 |    0.8809
    2 |   0.2161 |      0.7247 |     0.3896 | 0.2441 |           1.000000 |      9.3747 |       0.0340 |    0.9059
    3 |   0.1385 |      0.6950 |     0.5099 | 0.3931 |           1.000000 |      7.1821 |       0.0085 |    0.8780
    4 |   0.1210 |      0.6905 |     0.5061 | 0.5240 |           1.000000 |      7.1096 |       0.0085 |    0.9369
    5 |   0.0939 |      0.6903 |     0.5617 | 0.6341 |           1.000000 |      6.2119 |       0.0128 |    0.9109
    6 |   0.1014 |      0.6920 |     0.5059 | 0.7234 |           1.000000 |      6.6069 |       0.0043 |    0.9351
    7 |   0.0679 |      0.6914 |     0.5103 | 0.7937 |      

In [39]:
# ADOPT V4 FROM SOURCE-ONLY EVIDENCE, VERIFY DANN CLASSES, THEN TRAIN V4 CDAN

import csv
import hashlib
import importlib
import json
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score

CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

PROTOCOL_PATH = (
    CODE_ROOT / "shared/splits/pacs_sketch_seed6304.json"
)
INITIALIZATION = (
    V4_ROOT / "initialization/resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    CODE_ROOT
    / "task2/preregistration/DAN_STRENGTH_EXPECTATION.txt"
)

DANN_DIR = V4_ROOT / "dann"
DANN_RUN_RECORD = DANN_DIR / "run.json"
DANN_CHECKPOINT = DANN_DIR / "best.pt"

CDAN_DIR = V4_ROOT / "cdan"
CDAN_RUN_RECORD = CDAN_DIR / "run.json"
CDAN_LAST_CHECKPOINT = CDAN_DIR / "last.pt"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable.")

if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError("The v4 comparison requires a Tesla T4.")

for required_path in (
    CODE_ROOT,
    PACS_ROOT,
    PROTOCOL_PATH,
    INITIALIZATION,
    PREREGISTRATION,
    DANN_RUN_RECORD,
    DANN_CHECKPOINT,
    V4_ROOT / "experiment_lock.json",
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

# Ensure this process imports the v4 code.
for module_name in list(sys.modules):
    if (
        module_name == "shared"
        or module_name.startswith("shared.")
        or module_name == "task2"
        or module_name.startswith("task2.")
    ):
        del sys.modules[module_name]

sys.path = [
    entry
    for entry in sys.path
    if not (
        isinstance(entry, str)
        and entry.startswith("/content/task2")
    )
]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()

from shared.pacs import (
    CLASSES,
    SOURCES,
    load_protocol,
    make_validation_loader,
)
from task2.model import PACSClassifier

device = torch.device("cuda")
protocol = load_protocol(PROTOCOL_PATH)

# ------------------------------------------------------------
# 1. Source-only diagnostic of the selected v4 DANN checkpoint
# ------------------------------------------------------------

dann_record = json.loads(DANN_RUN_RECORD.read_text())
dann_checkpoint = torch.load(
    DANN_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert dann_record["target_labels_used"] is False
assert dann_checkpoint["target_labels_used"] is False

model = PACSClassifier(pretrained=False)
model.load_state_dict(dann_checkpoint["model_state"])
model.to(device)
model.eval()

source_diagnostic = {}
recomputed_f1_values = []
pooled_predictions = Counter()

with torch.inference_mode():
    for domain in SOURCES:
        loader = make_validation_loader(
            PACS_ROOT,
            protocol["source_splits"][domain]["validation"],
            batch_size=64,
            num_workers=2,
            pin_memory=True,
        )

        truth = []
        predictions = []

        for images, labels, _ in loader:
            logits, _ = model(
                images.to(device, non_blocking=True)
            )
            truth.extend(int(value) for value in labels.tolist())
            predictions.extend(
                int(value)
                for value in logits.argmax(dim=1).cpu().tolist()
            )

        domain_f1 = float(
            f1_score(
                truth,
                predictions,
                labels=list(range(7)),
                average="macro",
                zero_division=0,
            )
        )
        domain_accuracy = float(
            accuracy_score(truth, predictions)
        )

        prediction_counts = {
            CLASSES[class_id]: int(
                sum(prediction == class_id for prediction in predictions)
            )
            for class_id in range(7)
        }

        per_class_accuracy = {}
        for class_id, class_name in enumerate(CLASSES):
            indices = [
                index
                for index, label in enumerate(truth)
                if label == class_id
            ]
            per_class_accuracy[class_name] = float(
                np.mean(
                    [
                        predictions[index] == class_id
                        for index in indices
                    ]
                )
            )

        pooled_predictions.update(predictions)
        recomputed_f1_values.append(domain_f1)

        source_diagnostic[domain] = {
            "accuracy": domain_accuracy,
            "macro_f1": domain_f1,
            "prediction_counts": prediction_counts,
            "per_class_accuracy": per_class_accuracy,
        }

recomputed_mean_f1 = float(np.mean(recomputed_f1_values))
recorded_mean_f1 = float(
    dann_record["best_mean_source_macro_f1"]
)

if not np.isclose(
    recomputed_mean_f1,
    recorded_mean_f1,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Recomputed DANN source F1 does not match the run record."
    )

pooled_prediction_counts = {
    CLASSES[class_id]: int(pooled_predictions[class_id])
    for class_id in range(7)
}

predicted_classes = {
    class_id
    for class_id, count in pooled_predictions.items()
    if count > 0
}

if predicted_classes != set(range(7)):
    raise RuntimeError(
        "V4 DANN still omits one or more source classes; "
        "stop before adopting it."
    )

source_diagnostic["pooled_prediction_counts"] = (
    pooled_prediction_counts
)
source_diagnostic["mean_source_macro_f1"] = (
    recomputed_mean_f1
)
source_diagnostic["target_labels_accessed"] = False

diagnostic_path = (
    V4_ROOT / "dann_source_checkpoint_diagnostic.json"
)
diagnostic_path.write_text(
    json.dumps(source_diagnostic, indent=2) + "\n"
)

print("V4 DANN SELECTED-CHECKPOINT SOURCE DIAGNOSTIC")

for domain in SOURCES:
    result = source_diagnostic[domain]
    print(f"\n{domain}")
    print("  Accuracy:", result["accuracy"])
    print("  Macro-F1:", result["macro_f1"])
    print(
        "  Prediction counts:",
        result["prediction_counts"],
    )
    print(
        "  Per-class accuracy:",
        result["per_class_accuracy"],
    )

print(
    "\nPooled source prediction counts:",
    pooled_prediction_counts,
)
print("All seven classes predicted:", True)
print("Recomputed mean source macro-F1:", recomputed_mean_f1)
print("Target labels accessed:", False)

# Release diagnostic GPU memory before CDAN.
del model
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. Record the source-only v4 adoption decision
# ------------------------------------------------------------

with (DANN_DIR / "history.csv").open(newline="") as handle:
    dann_history = list(csv.DictReader(handle))

v3_metrics = {
    "best_mean_source_macro_f1": 0.6049088767773435,
    "maximum_pre_clipping_gradient_norm": 40455498.998820186,
    "mean_clipped_step_fraction": 0.9340425531914893,
    "checkpoint_sha256": (
        "321e3389a246ad179d8219584d04f9b089"
        "c8a13d7ef5512d34bceab4f0f676e4"
    ),
}

v4_metrics = {
    "best_mean_source_macro_f1": recorded_mean_f1,
    "maximum_pre_clipping_gradient_norm": max(
        float(row["gradient_norm"])
        for row in dann_history
    ),
    "mean_clipped_step_fraction": float(
        np.mean(
            [
                float(row["gradient_clipped_fraction"])
                for row in dann_history
            ]
        )
    ),
    "checkpoint_sha256": (
        dann_record["best_checkpoint_sha256"]
    ),
    "all_seven_source_classes_predicted": True,
}

github_record = json.loads(
    (V4_ROOT / "github_commit.json").read_text()
)

adoption_record = {
    "decision": (
        "adopt_v4_adversarial_feature_normalization_"
        "for_dann_and_cdan"
    ),
    "decision_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "basis": "source_information_only",
    "student_authorization": (
        "The student approved trying the documented approach "
        "and instructed fallback to v3 only if it produced no "
        "meaningful improvement."
    ),
    "v3_dann": v3_metrics,
    "v4_dann": v4_metrics,
    "source_f1_change": (
        recorded_mean_f1
        - v3_metrics["best_mean_source_macro_f1"]
    ),
    "unchanged_settings": [
        "GRL schedule",
        "domain-loss weight",
        "discriminator architecture",
        "AdamW settings",
        "global L2 clipping at 20",
        "common initialization",
        "source split",
        "sampling",
        "augmentation",
        "early stopping",
        "source-only checkpoint selection",
    ],
    "github_training_commit": github_record["commit"],
    "manifest_sha256": github_record["manifest_sha256"],
    "target_labels_accessed": False,
}

(V4_ROOT / "V4_ADOPTION_DECISION.json").write_text(
    json.dumps(adoption_record, indent=2) + "\n"
)

(V4_ROOT / "V4_ADOPTION_DECISION.md").write_text(
    f"""# Task 2 v4 adoption decision

The v4 adversarial-input normalization rule is adopted for DANN and
CDAN using source information only.

- V3 DANN source macro-F1: {v3_metrics['best_mean_source_macro_f1']:.10f}
- V4 DANN source macro-F1: {v4_metrics['best_mean_source_macro_f1']:.10f}
- Source F1 change: {adoption_record['source_f1_change']:.10f}
- V3 maximum pre-clipping gradient norm: {v3_metrics['maximum_pre_clipping_gradient_norm']:.4f}
- V4 maximum pre-clipping gradient norm: {v4_metrics['maximum_pre_clipping_gradient_norm']:.4f}
- V3 clipped-step fraction: {v3_metrics['mean_clipped_step_fraction']:.6f}
- V4 clipped-step fraction: {v4_metrics['mean_clipped_step_fraction']:.6f}
- All seven source classes predicted by selected v4 checkpoint: yes
- Target labels accessed: no
- Training-code commit: `{github_record['commit']}`

The classifier continues to use raw features. Only the DANN/CDAN
domain-discriminator input uses per-example L2-normalized features.
All other locked settings remain unchanged.
"""
)

print("\nV4 ADOPTION RECORDED")
print("Decision:", adoption_record["decision"])
print("Basis:", adoption_record["basis"])
print("Target labels accessed:", False)

# ------------------------------------------------------------
# 3. Train CDAN under the adopted v4 protocol
# ------------------------------------------------------------

if (
    CDAN_DIR.exists()
    and not CDAN_RUN_RECORD.exists()
    and not CDAN_LAST_CHECKPOINT.exists()
):
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    preserved = (
        V4_ROOT
        / f"cdan_incomplete_before_epoch_{timestamp}"
    )
    shutil.move(str(CDAN_DIR), str(preserved))
    print("Preserved incomplete pre-epoch CDAN at:", preserved)

if CDAN_RUN_RECORD.exists():
    print("\nV4 CDAN is already complete; verifying it.")
else:
    command = [
        sys.executable,
        "-m",
        "task2.train",
        "--run-id",
        "cdan",
        "--pacs-root",
        str(PACS_ROOT),
        "--dataset-source",
        (
            "Verified PACS archive; SHA256="
            "0dc9d0176fa27c9b4504e7c2e962aebe"
            "6a79ed0c1819b84148786e590f87e102"
        ),
        "--protocol",
        str(PROTOCOL_PATH),
        "--initialization",
        str(INITIALIZATION),
        "--preregistration",
        str(PREREGISTRATION),
        "--output",
        str(V4_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
    ]

    if CDAN_LAST_CHECKPOINT.exists():
        command.append("--resume")
        print("\nResuming official v4 CDAN.")
    else:
        print(
            "\nStarting official v4 CDAN from the "
            "locked common initialization."
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print("Target labels accessed:", False)

    subprocess.run(
        command,
        cwd=CODE_ROOT,
        check=True,
    )

if not CDAN_RUN_RECORD.is_file():
    raise RuntimeError(
        "CDAN ended without a completed run record."
    )

cdan_record = json.loads(CDAN_RUN_RECORD.read_text())

if cdan_record["target_labels_used"] is not False:
    raise RuntimeError(
        "CDAN record indicates target-label access."
    )

with (CDAN_DIR / "history.csv").open(newline="") as handle:
    cdan_history = list(csv.DictReader(handle))

if len(cdan_history) != cdan_record["epochs_completed"]:
    raise RuntimeError(
        "CDAN history length and run record disagree."
    )

post_norms = [
    float(row["adversarial_feature_norm_after"])
    for row in cdan_history
]

if not all(
    np.isfinite(value) and abs(value - 1.0) < 1e-5
    for value in post_norms
):
    raise RuntimeError(
        "CDAN discriminator-input normalization failed."
    )

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

cdan_checkpoint_sha256 = file_sha256(
    CDAN_DIR / "best.pt"
)

if (
    cdan_checkpoint_sha256
    != cdan_record["best_checkpoint_sha256"]
):
    raise RuntimeError("CDAN checkpoint hash mismatch.")

best_epoch = int(cdan_record["best_epoch"])
selected_row = next(
    row
    for row in cdan_history
    if int(row["epoch"]) == best_epoch
)

print("\nV4 CDAN SOURCE-ONLY TRAINING HISTORY")
print(
    "epoch | cls_loss | domain_loss | domain_acc | "
    "GRL | norm_after | grad_before | clipped_frac | source_F1"
)

for row in cdan_history:
    print(
        f"{int(row['epoch']):5d} | "
        f"{float(row['classification_loss']):8.4f} | "
        f"{float(row['domain_loss']):11.4f} | "
        f"{float(row['domain_accuracy']):10.4f} | "
        f"{float(row['grl_strength']):5.4f} | "
        f"{float(row['adversarial_feature_norm_after']):10.6f} | "
        f"{float(row['gradient_norm']):11.4f} | "
        f"{float(row['gradient_clipped_fraction']):12.4f} | "
        f"{float(row['mean_source_macro_f1']):9.4f}"
    )

print("\nV4 CDAN VERIFIED COMPLETE")
print("Epochs completed:", cdan_record["epochs_completed"])
print("Selected epoch:", best_epoch)
print(
    "Best source-validation macro-F1:",
    cdan_record["best_mean_source_macro_f1"],
)
print(
    "Maximum pre-clipping gradient norm:",
    max(
        float(row["gradient_norm"])
        for row in cdan_history
    ),
)
print(
    "Mean clipped-step fraction:",
    float(
        np.mean(
            [
                float(row["gradient_clipped_fraction"])
                for row in cdan_history
            ]
        )
    ),
)
print(
    "Selected-epoch domain accuracy:",
    float(selected_row["domain_accuracy"]),
)
print(
    "Selected-epoch GRL strength:",
    float(selected_row["grl_strength"]),
)
print(
    "Mean discriminator-input norm:",
    float(np.mean(post_norms)),
)
print("Checkpoint SHA256:", cdan_checkpoint_sha256)
print("Target labels accessed:", False)

V4 DANN SELECTED-CHECKPOINT SOURCE DIAGNOSTIC

photo
  Accuracy: 0.9670658682634731
  Macro-F1: 0.9645332198912147
  Prediction counts: {'dog': 41, 'elephant': 38, 'giraffe': 35, 'guitar': 36, 'horse': 36, 'house': 58, 'person': 90}
  Per-class accuracy: {'dog': 0.9736842105263158, 'elephant': 0.926829268292683, 'giraffe': 0.9722222222222222, 'guitar': 0.972972972972973, 'horse': 0.875, 'house': 1.0, 'person': 1.0}

art_painting
  Accuracy: 0.9097560975609756
  Macro-F1: 0.9105336987673406
  Prediction counts: {'dog': 81, 'elephant': 50, 'giraffe': 52, 'guitar': 38, 'horse': 31, 'house': 59, 'person': 99}
  Per-class accuracy: {'dog': 0.9210526315789473, 'elephant': 0.9019607843137255, 'giraffe': 0.8771929824561403, 'guitar': 0.972972972972973, 'horse': 0.75, 'house': 0.9491525423728814, 'person': 0.9444444444444444}

cartoon
  Accuracy: 0.9424307036247335
  Macro-F1: 0.9475555113060642
  Prediction counts: {'dog': 76, 'elephant': 84, 'giraffe': 66, 'guitar': 28, 'horse': 68, 'house': 

In [41]:
# FREEZE THE SIX OFFICIAL CHECKPOINTS — SOURCE DATA ONLY

import csv
import hashlib
import importlib
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score

V3_CODE_ROOT = Path("/content/task2-corrected-normalized-v3")
V4_CODE_ROOT = Path("/content/task2-adversarial-normalized-v4")

V3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_corrected_normalized_v3_20260923"
)
V4_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_adversarial_normalized_v4_20260923"
)

PACS_ROOT = Path("/content/atml_pacs/pacs/images")
PROTOCOL_PATH = (
    V4_CODE_ROOT
    / "shared/splits/pacs_sketch_seed6304.json"
)
INITIALIZATION = (
    V4_ROOT
    / "initialization/resnet18_v1_seed6304_common.pt"
)

FREEZE_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_official_freeze_20260923"
)
FREEZE_ROOT.mkdir(parents=True, exist_ok=True)

FREEZE_PATH = FREEZE_ROOT / "task2_checkpoint_freeze.json"
AUDIT_PATH = FREEZE_ROOT / "source_checkpoint_audit.json"

EXPECTED = {
    "source_only": {
        "version": "v3",
        "root": V3_ROOT,
        "best_epoch": 4,
        "source_f1": 0.9426262342459099,
        "checkpoint_sha256": (
            "3d28a223e4b97b323cb3a20dcb5b7577"
            "af96631f2e6ef1f2bc99d53d85761327"
        ),
    },
    "dan_0p1": {
        "version": "v3",
        "root": V3_ROOT,
        "best_epoch": 2,
        "source_f1": 0.9320428032141853,
        "checkpoint_sha256": (
            "4d0d7132d25404bfab98446ad49e11669"
            "5d90cc1f2d6a7750514de4da14d4499"
        ),
    },
    "dan_1": {
        "version": "v3",
        "root": V3_ROOT,
        "best_epoch": 10,
        "source_f1": 0.9351645184948074,
        "checkpoint_sha256": (
            "a3274833ee4cf143a1e8a25eb340da431"
            "422d1d264e52a3c82f75d55a45e9563"
        ),
    },
    "dan_10": {
        "version": "v3",
        "root": V3_ROOT,
        "best_epoch": 1,
        "source_f1": 0.05066996495567924,
        "checkpoint_sha256": (
            "487dc643227bae3f9057ccb7f05a282a2"
            "52cadbd9471d38dbb78fbb6d30ea5aa"
        ),
    },
    "dann": {
        "version": "v4",
        "root": V4_ROOT,
        "best_epoch": 9,
        "source_f1": 0.9408741433215398,
        "checkpoint_sha256": (
            "1154ba6ef175f7ba728f16697f199db65"
            "a574c7a4e4e7e3613db22e4880443ee"
        ),
    },
    "cdan": {
        "version": "v4",
        "root": V4_ROOT,
        "best_epoch": 6,
        "source_f1": 0.9440964260445351,
        "checkpoint_sha256": (
            "93364e2cd39a4865277c60165c4d7a77"
            "b946525936d4b1c796ad2bdcd0bd14c4"
        ),
    },
}

# Use the unchanged v4 model and source-data pipeline.
for module_name in list(sys.modules):
    if (
        module_name == "shared"
        or module_name.startswith("shared.")
        or module_name == "task2"
        or module_name.startswith("task2.")
    ):
        del sys.modules[module_name]

sys.path = [
    entry
    for entry in sys.path
    if not (
        isinstance(entry, str)
        and entry.startswith("/content/task2")
    )
]
sys.path.insert(0, str(V4_CODE_ROOT))
importlib.invalidate_caches()

from shared.pacs import (
    CLASSES,
    SOURCES,
    load_protocol,
    make_validation_loader,
)
from task2.model import PACSClassifier

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

for required_path in (
    V3_CODE_ROOT,
    V4_CODE_ROOT,
    V3_ROOT,
    V4_ROOT,
    PACS_ROOT,
    PROTOCOL_PATH,
    INITIALIZATION,
    V4_ROOT / "V4_ADOPTION_DECISION.json",
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

# Confirm model architecture and data code are identical between v3/v4.
v3_manifest = json.loads(
    (
        V3_CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
    ).read_text()
)
v4_manifest = json.loads(
    (
        V4_CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
    ).read_text()
)

def manifested_hash(manifest, relative_path):
    return next(
        item["sha256"]
        for item in manifest["files"]
        if item["path"] == relative_path
    )

for unchanged_file in (
    "task2/model.py",
    "shared/pacs.py",
    "shared/splits/pacs_sketch_seed6304.json",
):
    if (
        manifested_hash(v3_manifest, unchanged_file)
        != manifested_hash(v4_manifest, unchanged_file)
    ):
        raise RuntimeError(
            f"Unexpected v3/v4 difference: {unchanged_file}"
        )

protocol = load_protocol(PROTOCOL_PATH)
device = torch.device("cuda")

validation_loaders = {
    domain: make_validation_loader(
        PACS_ROOT,
        protocol["source_splits"][domain]["validation"],
        batch_size=64,
        num_workers=2,
        pin_memory=True,
    )
    for domain in SOURCES
}

initialization_payload = torch.load(
    INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)
initialization_state = initialization_payload["state_dict"]

initialization_bn = {
    name: value
    for name, value in initialization_state.items()
    if name.endswith(
        (
            "running_mean",
            "running_var",
            "num_batches_tracked",
        )
    )
}

audited_runs = {}
shared_identities = []

for run_id, specification in EXPECTED.items():
    run_dir = specification["root"] / run_id
    run_record_path = run_dir / "run.json"
    checkpoint_path = run_dir / "best.pt"
    history_path = run_dir / "history.csv"

    for path in (
        run_record_path,
        checkpoint_path,
        history_path,
        run_dir / "best_source_validation.json",
    ):
        if not path.is_file():
            raise FileNotFoundError(path)

    run_record = json.loads(run_record_path.read_text())
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if run_record["run_id"] != run_id:
        raise RuntimeError(f"Wrong run identity: {run_id}")

    if run_record["target_labels_used"] is not False:
        raise RuntimeError(f"Target-label flag failed: {run_id}")

    if checkpoint["target_labels_used"] is not False:
        raise RuntimeError(
            f"Checkpoint target-label flag failed: {run_id}"
        )

    actual_checkpoint_sha256 = file_sha256(checkpoint_path)

    if (
        actual_checkpoint_sha256
        != specification["checkpoint_sha256"]
        or actual_checkpoint_sha256
        != run_record["best_checkpoint_sha256"]
    ):
        raise RuntimeError(
            f"Checkpoint SHA256 mismatch: {run_id}"
        )

    if int(run_record["best_epoch"]) != specification["best_epoch"]:
        raise RuntimeError(f"Best epoch mismatch: {run_id}")

    if int(checkpoint["epoch"]) != specification["best_epoch"]:
        raise RuntimeError(
            f"Checkpoint epoch mismatch: {run_id}"
        )

    if checkpoint["identity"] != run_record["identity"]:
        raise RuntimeError(
            f"Checkpoint/run identity mismatch: {run_id}"
        )

    if not np.isclose(
        float(run_record["best_mean_source_macro_f1"]),
        specification["source_f1"],
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"Recorded source F1 mismatch: {run_id}"
        )

    with history_path.open(newline="") as handle:
        history = list(csv.DictReader(handle))

    if len(history) != int(run_record["epochs_completed"]):
        raise RuntimeError(
            f"History-length mismatch: {run_id}"
        )

    history_f1 = [
        float(row["mean_source_macro_f1"])
        for row in history
    ]
    reconstructed_best_index = int(np.argmax(history_f1))
    reconstructed_best_epoch = int(
        history[reconstructed_best_index]["epoch"]
    )

    if reconstructed_best_epoch != specification["best_epoch"]:
        raise RuntimeError(
            f"Checkpoint-selection reconstruction failed: {run_id}"
        )

    if not np.isclose(
        history_f1[reconstructed_best_index],
        specification["source_f1"],
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"Best history F1 mismatch: {run_id}"
        )

    # Every completed run stopped after five epochs without improvement.
    expected_completed_epochs = (
        specification["best_epoch"] + 5
    )
    if int(run_record["epochs_completed"]) != expected_completed_epochs:
        raise RuntimeError(
            f"Early-stopping reconstruction failed: {run_id}"
        )

    # Verify frozen BatchNorm buffers directly from the checkpoint.
    checkpoint_state = checkpoint["model_state"]

    for name, expected_buffer in initialization_bn.items():
        if not torch.equal(
            checkpoint_state[name].cpu(),
            expected_buffer.cpu(),
        ):
            raise RuntimeError(
                f"BatchNorm buffer changed in {run_id}: {name}"
            )

    # Independently recompute source-validation metrics and predictions.
    model = PACSClassifier(pretrained=False)
    model.load_state_dict(checkpoint_state)
    model.to(device)
    model.eval()

    domain_results = {}
    recomputed_domain_f1 = []
    pooled_counts = Counter()

    with torch.inference_mode():
        for domain in SOURCES:
            truth = []
            predictions = []

            for images, labels, _ in validation_loaders[domain]:
                logits, _ = model(
                    images.to(device, non_blocking=True)
                )
                truth.extend(
                    int(value) for value in labels.tolist()
                )
                predictions.extend(
                    int(value)
                    for value
                    in logits.argmax(dim=1).cpu().tolist()
                )

            accuracy = float(
                accuracy_score(truth, predictions)
            )
            macro_f1 = float(
                f1_score(
                    truth,
                    predictions,
                    labels=list(range(7)),
                    average="macro",
                    zero_division=0,
                )
            )

            recorded_validation = checkpoint[
                "source_validation"
            ]

            if not np.isclose(
                accuracy,
                float(
                    recorded_validation[
                        f"{domain}_accuracy"
                    ]
                ),
                rtol=0.0,
                atol=1e-12,
            ):
                raise RuntimeError(
                    f"Source accuracy mismatch: {run_id}/{domain}"
                )

            if not np.isclose(
                macro_f1,
                float(
                    recorded_validation[
                        f"{domain}_macro_f1"
                    ]
                ),
                rtol=0.0,
                atol=1e-12,
            ):
                raise RuntimeError(
                    f"Source F1 mismatch: {run_id}/{domain}"
                )

            counts = {
                CLASSES[class_id]: int(
                    sum(
                        prediction == class_id
                        for prediction in predictions
                    )
                )
                for class_id in range(7)
            }

            per_class_accuracy = {}
            for class_id, class_name in enumerate(CLASSES):
                indices = [
                    index
                    for index, label in enumerate(truth)
                    if label == class_id
                ]
                per_class_accuracy[class_name] = float(
                    np.mean(
                        [
                            predictions[index] == class_id
                            for index in indices
                        ]
                    )
                )

            pooled_counts.update(predictions)
            recomputed_domain_f1.append(macro_f1)

            domain_results[domain] = {
                "accuracy": accuracy,
                "macro_f1": macro_f1,
                "prediction_counts": counts,
                "per_class_accuracy": per_class_accuracy,
            }

    recomputed_mean_f1 = float(
        np.mean(recomputed_domain_f1)
    )

    if not np.isclose(
        recomputed_mean_f1,
        specification["source_f1"],
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"Recomputed mean F1 mismatch: {run_id}"
        )

    identity = run_record["identity"]
    shared_identities.append(
        {
            key: value
            for key, value in identity.items()
            if key not in {"config", "code_sha256"}
        }
    )

    audited_runs[run_id] = {
        "protocol_version": specification["version"],
        "run_directory": str(run_dir),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": actual_checkpoint_sha256,
        "selected_epoch": specification["best_epoch"],
        "epochs_completed": int(
            run_record["epochs_completed"]
        ),
        "mean_source_validation_macro_f1": (
            recomputed_mean_f1
        ),
        "source_domains": domain_results,
        "pooled_source_prediction_counts": {
            CLASSES[class_id]: int(pooled_counts[class_id])
            for class_id in range(7)
        },
        "identity": identity,
        "batchnorm_buffers_unchanged": True,
        "checkpoint_selection_reconstructed": True,
        "target_labels_accessed": False,
    }

    print(
        f"{run_id}: VERIFIED — "
        f"version={specification['version']}, "
        f"epoch={specification['best_epoch']}, "
        f"source_F1={recomputed_mean_f1:.10f}"
    )

    del model
    torch.cuda.empty_cache()

# All non-code/config identities must agree across v3 and v4.
reference_identity = shared_identities[0]
for identity in shared_identities[1:]:
    if identity != reference_identity:
        raise RuntimeError(
            "Cross-run dataset, split, initialization, "
            "environment, or preregistration identity differs."
        )

source_audit = {
    "status": "SOURCE_CHECKPOINT_AUDIT_PASS",
    "runs": audited_runs,
    "common_identity": reference_identity,
    "model_and_data_code_equal_between_v3_v4": True,
    "target_labels_accessed": False,
}

audit_text = json.dumps(source_audit, indent=2) + "\n"

if AUDIT_PATH.exists():
    if AUDIT_PATH.read_text() != audit_text:
        raise RuntimeError(
            "Existing source audit differs; refusing overwrite."
        )
else:
    AUDIT_PATH.write_text(audit_text)

v3_commit_record = (
    json.loads(
        (V3_ROOT / "github_commit.json").read_text()
    )
    if (V3_ROOT / "github_commit.json").exists()
    else {
        "commit": "b96f184",
        "note": "Previously verified v3 training commit",
    }
)
v4_commit_record = json.loads(
    (V4_ROOT / "github_commit.json").read_text()
)
v4_adoption = json.loads(
    (V4_ROOT / "V4_ADOPTION_DECISION.json").read_text()
)

freeze = {
    "status": "FROZEN_BEFORE_TARGET_LABEL_ACCESS",
    "date": "2026-09-23",
    "official_runs": {
        run_id: {
            "protocol_version": result["protocol_version"],
            "checkpoint_path": result["checkpoint_path"],
            "checkpoint_sha256": result[
                "checkpoint_sha256"
            ],
            "selected_epoch": result["selected_epoch"],
            "mean_source_validation_macro_f1": result[
                "mean_source_validation_macro_f1"
            ],
        }
        for run_id, result in audited_runs.items()
    },
    "v3_training_commit": v3_commit_record,
    "v4_training_commit": v4_commit_record,
    "v4_adoption_decision": v4_adoption,
    "common_identity": reference_identity,
    "source_audit_path": str(AUDIT_PATH),
    "source_audit_sha256": file_sha256(AUDIT_PATH),
    "checkpoint_selection_rule": (
        "strict improvement in unweighted mean source-validation "
        "macro-F1; earliest exact tie; patience five"
    ),
    "target_labels_accessed": False,
    "changes_after_freeze": (
        "No checkpoint, method version, hyperparameter, split, "
        "or selection change is permitted after target evaluation."
    ),
}

freeze_text = json.dumps(freeze, indent=2) + "\n"

if FREEZE_PATH.exists():
    if FREEZE_PATH.read_text() != freeze_text:
        raise RuntimeError(
            "Existing freeze differs; refusing overwrite."
        )
else:
    FREEZE_PATH.write_text(freeze_text)

freeze_sha256 = file_sha256(FREEZE_PATH)
(FREEZE_ROOT / "task2_checkpoint_freeze.sha256").write_text(
    freeze_sha256 + "  task2_checkpoint_freeze.json\n"
)

print("\nOFFICIAL TASK 2 CHECKPOINT FREEZE COMPLETE")
print("Frozen runs:", len(audited_runs))
print("Source audit:", AUDIT_PATH)
print("Source audit SHA256:", file_sha256(AUDIT_PATH))
print("Freeze file:", FREEZE_PATH)
print("Freeze SHA256:", freeze_sha256)
print("Target labels accessed:", False)
print(
    "Checkpoint or protocol changes after target evaluation:",
    "PROHIBITED",
)

source_only: VERIFIED — version=v3, epoch=4, source_F1=0.9426262342
dan_0p1: VERIFIED — version=v3, epoch=2, source_F1=0.9320428032
dan_1: VERIFIED — version=v3, epoch=10, source_F1=0.9351645185
dan_10: VERIFIED — version=v3, epoch=1, source_F1=0.0506699650
dann: VERIFIED — version=v4, epoch=9, source_F1=0.9408741433
cdan: VERIFIED — version=v4, epoch=6, source_F1=0.9440964260

OFFICIAL TASK 2 CHECKPOINT FREEZE COMPLETE
Frozen runs: 6
Source audit: /content/drive/MyDrive/ATML-PA1/task2_official_freeze_20260923/source_checkpoint_audit.json
Source audit SHA256: 4567df8de5d2b86d662c8c38017df1d0004b626b461c793d767207422033c1ca
Freeze file: /content/drive/MyDrive/ATML-PA1/task2_official_freeze_20260923/task2_checkpoint_freeze.json
Freeze SHA256: baacc896c12285216eee120b785475b69c1e3bfcee911c33a759a72c897f15b5
Target labels accessed: False
Checkpoint or protocol changes after target evaluation: PROHIBITED


In [42]:
# OFFICIAL TASK 2 FINAL EVALUATION — FIRST CORRECTED-RUN SKETCH LABEL ACCESS

from google.colab import drive
drive.mount("/content/drive")

import csv
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY = (
    "https://github.com/"
    "therealshaheer11-glithc/ATML-Assignment-1.git"
)

EVAL_CODE_ROOT = Path("/content/task2-final-evaluation-github")
PACS_ROOT = Path("/content/atml_pacs/pacs/images")

FREEZE_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_official_freeze_20260923"
)
FREEZE_PATH = (
    FREEZE_ROOT / "task2_checkpoint_freeze.json"
)
SOURCE_AUDIT_PATH = (
    FREEZE_ROOT / "source_checkpoint_audit.json"
)

RESULT_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task2_final_evaluation_20260923"
)
PARTIAL_ROOT = RESULT_ROOT.with_name(
    RESULT_ROOT.name + ".partial"
)

EXPECTED_FREEZE_SHA256 = (
    "baacc896c12285216eee120b785475b69"
    "c1e3bfcee911c33a759a72c897f15b5"
)
EXPECTED_TRAINING_MANIFEST_SHA256 = (
    "3cec075fea3295ece5ff42ca518424723"
    "497b95995bd58e8840a0bc13584c01c"
)
EXPECTED_EVALUATION_MANIFEST_SHA256 = (
    "39e703f62c6adb7d5e83a4bfbccefab8"
    "adc7530e2f145c3548ab515f208d53d7"
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

if not FREEZE_PATH.is_file():
    raise FileNotFoundError(FREEZE_PATH)

if not SOURCE_AUDIT_PATH.is_file():
    raise FileNotFoundError(SOURCE_AUDIT_PATH)

if not PACS_ROOT.is_dir():
    raise FileNotFoundError(PACS_ROOT)

if file_sha256(FREEZE_PATH) != EXPECTED_FREEZE_SHA256:
    raise RuntimeError(
        "The official checkpoint freeze hash has changed."
    )

# Refresh only the temporary GitHub checkout.
if EVAL_CODE_ROOT.exists():
    shutil.rmtree(EVAL_CODE_ROOT)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        REPOSITORY,
        str(EVAL_CODE_ROOT),
    ],
    check=True,
)

evaluation_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(EVAL_CODE_ROOT),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

training_manifest_path = (
    EVAL_CODE_ROOT / "task2/PACKAGE-MANIFEST.json"
)
evaluation_manifest_path = (
    EVAL_CODE_ROOT
    / "task2/FINAL-EVALUATION-MANIFEST.json"
)

if (
    file_sha256(training_manifest_path)
    != EXPECTED_TRAINING_MANIFEST_SHA256
):
    raise RuntimeError(
        "The committed v4 training manifest changed."
    )

if (
    file_sha256(evaluation_manifest_path)
    != EXPECTED_EVALUATION_MANIFEST_SHA256
):
    raise RuntimeError(
        "The committed final-evaluation manifest changed."
    )

# Verify every governed training and evaluation file.
for manifest_path in (
    training_manifest_path,
    evaluation_manifest_path,
):
    manifest = json.loads(manifest_path.read_text())

    for record in manifest["files"]:
        path = EVAL_CODE_ROOT / record["path"]

        if not path.is_file():
            raise FileNotFoundError(path)

        if path.stat().st_size != record["bytes"]:
            raise RuntimeError(
                f"Manifest size mismatch: {record['path']}"
            )

        if file_sha256(path) != record["sha256"]:
            raise RuntimeError(
                f"Manifest SHA256 mismatch: {record['path']}"
            )

# Compile all committed code.
subprocess.run(
    [
        sys.executable,
        "-m",
        "compileall",
        "-q",
        str(EVAL_CODE_ROOT),
    ],
    check=True,
)

# Run the nine training-protocol tests and five evaluation tests
# in a fresh process to avoid cached notebook imports.
test_program = r"""
import importlib.util
import sys
from pathlib import Path

root = Path.cwd()
sys.path.insert(0, str(root))

test_files = [
    root / "tests/test_locked_choices.py",
    root / "tests/test_final_evaluation.py",
]

total = 0

for file_index, test_file in enumerate(test_files):
    spec = importlib.util.spec_from_file_location(
        f"task2_test_module_{file_index}",
        test_file,
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    for name in sorted(dir(module)):
        if name.startswith("test_"):
            getattr(module, name)()
            total += 1

print("TOTAL TESTS PASSED:", total)

if total != 14:
    raise RuntimeError(
        f"Expected 14 tests, executed {total}"
    )
"""

subprocess.run(
    [sys.executable, "-c", test_program],
    cwd=EVAL_CODE_ROOT,
    check=True,
)

# Record the exact evaluator before target labels are accessed.
evaluation_code_lock = {
    "status": "EVALUATION_CODE_LOCKED_BEFORE_TARGET_ACCESS",
    "repository": REPOSITORY,
    "commit": evaluation_commit,
    "training_manifest_sha256": (
        EXPECTED_TRAINING_MANIFEST_SHA256
    ),
    "evaluation_manifest_sha256": (
        EXPECTED_EVALUATION_MANIFEST_SHA256
    ),
    "freeze_sha256": EXPECTED_FREEZE_SHA256,
    "target_labels_accessed": False,
}

(FREEZE_ROOT / "evaluation_code_lock.json").write_text(
    json.dumps(evaluation_code_lock, indent=2) + "\n"
)

if RESULT_ROOT.exists():
    complete_path = (
        RESULT_ROOT / "evaluation_complete.json"
    )

    if not complete_path.is_file():
        raise RuntimeError(
            "The final result directory exists without a "
            "completion record; preserve it and investigate."
        )

    print(
        "Official final evaluation is already complete; "
        "verifying saved results."
    )
else:
    # Preserve any interrupted evaluation. Repeating evaluation with
    # unchanged frozen checkpoints is permitted and changes no decision.
    if PARTIAL_ROOT.exists():
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%dT%H%M%SZ")
        preserved_partial = FREEZE_ROOT / (
            f"final_evaluation_partial_{timestamp}"
        )
        shutil.move(
            str(PARTIAL_ROOT),
            str(preserved_partial),
        )
        print(
            "Preserved interrupted evaluation at:",
            preserved_partial,
        )

    command = [
        sys.executable,
        "-m",
        "task2.evaluate_frozen",
        "--freeze",
        str(FREEZE_PATH),
        "--expected-freeze-sha256",
        EXPECTED_FREEZE_SHA256,
        "--source-audit",
        str(SOURCE_AUDIT_PATH),
        "--pacs-root",
        str(PACS_ROOT),
        "--protocol",
        str(
            EVAL_CODE_ROOT
            / "shared/splits/pacs_sketch_seed6304.json"
        ),
        "--output",
        str(RESULT_ROOT),
        "--device",
        "cuda",
        "--num-workers",
        "2",
    ]

    print("\nSTARTING OFFICIAL FINAL EVALUATION")
    print("Evaluation commit:", evaluation_commit)
    print("Freeze SHA256:", EXPECTED_FREEZE_SHA256)
    print("Training or checkpoint changes:", False)
    print(
        "Sketch labels will now be accessed only for "
        "frozen-checkpoint analysis."
    )

    subprocess.run(
        command,
        cwd=EVAL_CODE_ROOT,
        check=True,
    )

# Verify every published result artifact.
result_manifest_path = (
    RESULT_ROOT / "EVALUATION-MANIFEST.json"
)
completion_path = (
    RESULT_ROOT / "evaluation_complete.json"
)

if not result_manifest_path.is_file():
    raise FileNotFoundError(result_manifest_path)

if not completion_path.is_file():
    raise FileNotFoundError(completion_path)

result_manifest = json.loads(
    result_manifest_path.read_text()
)

if result_manifest["status"] != "COMPLETE":
    raise RuntimeError(
        "Final evaluation manifest is incomplete."
    )

for record in result_manifest["files"]:
    path = RESULT_ROOT / record["path"]

    if not path.is_file():
        raise FileNotFoundError(path)

    if path.stat().st_size != record["bytes"]:
        raise RuntimeError(
            f"Result size mismatch: {record['path']}"
        )

    if file_sha256(path) != record["sha256"]:
        raise RuntimeError(
            f"Result SHA256 mismatch: {record['path']}"
        )

completion = json.loads(completion_path.read_text())

if completion["status"] != "FINAL_EVALUATION_COMPLETE":
    raise RuntimeError("Evaluation did not complete.")

if completion["freeze_sha256"] != EXPECTED_FREEZE_SHA256:
    raise RuntimeError(
        "Evaluation used the wrong checkpoint freeze."
    )

if (
    completion["target_labels_used_for_training_or_selection"]
    is not False
):
    raise RuntimeError("Target-leakage flag failed.")

# Save the evaluation commit alongside the completed results.
(RESULT_ROOT / "evaluation_code_commit.json").write_text(
    json.dumps(
        {
            "repository": REPOSITORY,
            "commit": evaluation_commit,
            "training_manifest_sha256": (
                EXPECTED_TRAINING_MANIFEST_SHA256
            ),
            "evaluation_manifest_sha256": (
                EXPECTED_EVALUATION_MANIFEST_SHA256
            ),
            "freeze_sha256": EXPECTED_FREEZE_SHA256,
            "evaluation_complete": True,
            "target_labels_first_accessed_after_freeze": True,
            "target_labels_used_for_training_or_selection": False,
        },
        indent=2,
    )
    + "\n"
)

def read_csv(path):
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

main_rows = read_csv(
    RESULT_ROOT / "main_four_comparison.csv"
)
strength_rows = read_csv(
    RESULT_ROOT / "dan_strength_study.csv"
)
class_summary = json.loads(
    (
        RESULT_ROOT / "class_change_summary.json"
    ).read_text()
)

print("\nMAIN FOUR-METHOD COMPARISON")
print(
    "method | source_F1 | Sketch_acc | Sketch_F1 | "
    "change_vs_ERM | separability"
)

for row in main_rows:
    print(
        f"{row['method']:14s} | "
        f"{float(row['mean_source_macro_f1']):9.4f} | "
        f"{float(row['target_accuracy']):10.4f} | "
        f"{float(row['target_macro_f1']):9.4f} | "
        f"{float(row['target_accuracy_change_vs_source_only']):13.4f} | "
        f"{float(row['domain_separability']):12.4f}"
    )

print("\nDAN ALIGNMENT-STRENGTH STUDY")
print(
    "method | source_F1 | Sketch_acc | Sketch_F1 | "
    "separability"
)

for row in strength_rows:
    print(
        f"{row['method']:14s} | "
        f"{float(row['mean_source_macro_f1']):9.4f} | "
        f"{float(row['target_accuracy']):10.4f} | "
        f"{float(row['target_macro_f1']):9.4f} | "
        f"{float(row['domain_separability']):12.4f}"
    )

print("\nCLASS-CHANGE SUMMARY")
for run_id in ("dan_1", "dann", "cdan"):
    print(run_id, class_summary[run_id])

print("\nOFFICIAL TASK 2 FINAL EVALUATION COMPLETE")
print("Evaluation commit:", evaluation_commit)
print("Frozen checkpoints evaluated:", 6)
print("Sketch predictions per checkpoint:", 3929)
print("Result files verified:", len(result_manifest["files"]))
print("Results:", RESULT_ROOT)
print("Plots:", RESULT_ROOT / "plots")
print(
    "Target labels used for training or checkpoint selection:",
    False,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STARTING OFFICIAL FINAL EVALUATION
Evaluation commit: edecd5b9429799cc51c9e96625191beaf45562af
Freeze SHA256: baacc896c12285216eee120b785475b69c1e3bfcee911c33a759a72c897f15b5
Training or checkpoint changes: False
Sketch labels will now be accessed only for frozen-checkpoint analysis.

MAIN FOUR-METHOD COMPARISON
method | source_F1 | Sketch_acc | Sketch_F1 | change_vs_ERM | separability
Source-only    |    0.9426 |     0.6200 |    0.6574 |        0.0000 |       0.9959
DAN lambda=1   |    0.9352 |     0.7582 |    0.7076 |        0.1382 |       0.9011
DANN           |    0.9409 |     0.6908 |    0.7115 |        0.0708 |       0.9918
CDAN           |    0.9441 |     0.7203 |    0.7464 |        0.1003 |       0.9931

DAN ALIGNMENT-STRENGTH STUDY
method | source_F1 | Sketch_acc | Sketch_F1 | separability
Source-only    |    0.9426 |     0.6200 |    0.6574 |       